# Optimizacion de capacidad de un sistema WPEB con metaheuristicas

**Sistema WPEB**: viento (**W**ind) + fotovoltaica (**P**V) + **E**lectrolizador + **B**ateria.

Replicacion y extension discreta de:

> Li, J., Zhang, Y., Wang, S., et al. (2024). *Capacity optimization of a
> wind-photovoltaic-electrolysis-battery (WPEB) hybrid energy system for power and
> hydrogen generation.* Applied Energy.

Autor: Luciano Andres Oliva Jara.

---

## Que hay en este notebook

1. La **configuracion completa** del escenario y de las 6 metaheuristicas, en dataclasses
   explicitas y visibles (seccion 2 y 3).
2. El **modelo horario** completo: recurso eolico y solar, despacho hora a hora, economia
   (NPC, LCOE, LCOH). Nada se importa desde otros archivos `.py` (secciones 6 a 9).
3. La **validacion contra las cifras publicadas** por el paper (seccion 10).
4. El **barrido exhaustivo** de la instancia base, que da el optimo global por enumeracion
   (seccion 11).
5. Las **6 metaheuristicas programadas en el notebook**: GWO, PSO, GA, SHO, WOA y una
   busqueda aleatoria como linea base (secciones 12 a 14).
6. La **comparativa**: ranking, convergencia, dispersion y **tests estadisticos**
   (Wilcoxon pareado, tamano de efecto A12, correccion de Holm-Bonferroni)
   (secciones 16 a 18).
7. Un **estudio de sensibilidad** sobre las dos restricciones que definen el dominio
   factible (seccion 19).

## Como ejecutarlo

```text
notebooks/
  H2_HRES_trabajo.ipynb   <- este archivo
  src/
    hourly_2008.csv       <- el dato horario, 8784 filas (2008 fue bisiesto)
```

Requisitos: `numpy`, `pandas`, `matplotlib`, `scipy`. No hay que instalar ningun paquete
propio ni descargar nada de internet.

```bash
pip install numpy pandas matplotlib scipy
```

Luego **Run All**. La celda de la seccion 1 tiene una variable `MODO`:

| `MODO` | Presupuesto de la comparativa | Duracion aproximada |
|---|---|---|
| `"rapido"` (por defecto) | 3 semillas x 8 individuos x 8 iteraciones | ~1 minuto en total |
| `"completo"` | 30 semillas x 20 individuos x 30 iteraciones | ~15 minutos en total |

`"rapido"` sirve para revisar que todo corre y leer la estructura del trabajo. `"completo"`
es el experimento reportado: 600 evaluaciones por corrida, 30 semillas por algoritmo.

## 1. Configuracion de la ejecucion

Aqui se decide el presupuesto de computo y se resuelve la ruta del dato horario. Es la
unica celda que hace falta tocar para pasar de una revision rapida al experimento completo.

In [ ]:
import math
import time
from dataclasses import dataclass, field, fields, is_dataclass, replace
from enum import IntEnum
from itertools import combinations
from pathlib import Path
from typing import NamedTuple, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from IPython.display import Markdown, display

# --------------------------------------------------------------------------
# PRESUPUESTO DE COMPUTO. Cambiar esta linea es lo unico necesario para pasar
# de la revision rapida al experimento completo del trabajo.
# --------------------------------------------------------------------------
MODO = "rapido"          # "rapido" | "completo"

PRESUPUESTOS = {
    # semillas, individuos, iteraciones
    "rapido":   {"n_seeds": 3,  "population": 8,  "iterations": 8},
    "completo": {"n_seeds": 30, "population": 20, "iterations": 30},
}
if MODO not in PRESUPUESTOS:
    raise ValueError("MODO debe ser 'rapido' o 'completo', se recibio {!r}".format(MODO))
PRESUPUESTO = PRESUPUESTOS[MODO]

ALGORITMOS = ("gwo", "pso", "ga", "sho", "woa", "random")

# --------------------------------------------------------------------------
# Ruta del dato horario. Funciona tanto si el notebook se abre desde su propia
# carpeta como desde la raiz del repositorio.
# --------------------------------------------------------------------------
CWD = Path.cwd().resolve()
for candidate in (CWD, CWD / "notebooks", CWD.parent):
    if (candidate / "src" / "hourly_2008.csv").exists():
        NOTEBOOK_DIR = candidate
        break
else:
    raise FileNotFoundError(
        "No encuentro src/hourly_2008.csv. El notebook espera esta estructura:\n"
        "  H2_HRES_trabajo.ipynb\n"
        "  src/hourly_2008.csv"
    )
HOURLY_PATH = NOTEBOOK_DIR / "src" / "hourly_2008.csv"

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
plt.rcParams.update({
    "figure.figsize": (9, 5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

evaluaciones_por_corrida = PRESUPUESTO["population"] * PRESUPUESTO["iterations"]
evaluaciones_totales = evaluaciones_por_corrida * PRESUPUESTO["n_seeds"] * len(ALGORITMOS)

print("Dato horario      :", HOURLY_PATH)
print("Modo              :", MODO)
print("Presupuesto       : {} semillas x {} individuos x {} iteraciones".format(
    PRESUPUESTO["n_seeds"], PRESUPUESTO["population"], PRESUPUESTO["iterations"]))
print("Evaluaciones/corrida:", evaluaciones_por_corrida)
print("Evaluaciones totales:", evaluaciones_totales, "({} algoritmos)".format(len(ALGORITMOS)))
print("Tiempo estimado comparativa: ~{:.0f} s".format(evaluaciones_totales * 0.007))

## 2. Configuracion del escenario

Todo parametro fisico, economico y de restriccion vive en dataclasses congeladas
(`frozen=True`) con validacion en `__post_init__`. No hay diccionarios sueltos ni claves
magicas: una clave mal escrita falla al construir el escenario, no dentro del bucle de
optimizacion.

Las propiedades derivadas (`specific_consumption_kwh_per_kg`, `electrolyzer_max_mw`,
`eta_charge`) se calculan a partir de los parametros base para que no puedan
desincronizarse entre el modelo base y el extendido.

Los valores por defecto son **los del paper**: Damao Banner (Baotou, Mongolia Interior),
200 MW de generacion renovable, AGSR <= 20%, horizonte de 25 anos y tasa real de 4.35%.

In [ ]:
class ConfigError(ValueError):
    """Error de configuracion: clave desconocida, tipo o rango invalido."""


@dataclass(frozen=True)
class SiteConfig:
    """Emplazamiento y ventana temporal. Por defecto, el sitio del paper."""

    lat: float = 41.70
    lon: float = 110.43
    start_year: int = 2001
    end_year: int = 2021
    analysis_year: Optional[int] = 2008

    # NASA POWER entrega los datos en UTC, pero la complementariedad viento-solar
    # se mide por dia solar local. Damao Banner esta en UTC+8.
    utc_offset_hours: float = 8.0

    def __post_init__(self):
        if not -90.0 <= self.lat <= 90.0:
            raise ConfigError("lat fuera de rango [-90, 90]: " + str(self.lat))
        if not -180.0 <= self.lon <= 180.0:
            raise ConfigError("lon fuera de rango [-180, 180]: " + str(self.lon))
        if self.start_year > self.end_year:
            raise ConfigError("start_year > end_year")


@dataclass(frozen=True)
class WindConfig:
    """Turbina y recurso eolico.

    La curva de potencia es cubica simplificada entre cut-in y velocidad nominal.

    ``wind_speed_source`` elige que serie alimenta la curva:

    - ``"measured_50m"``: WS50M tal como la mide NASA POWER.
    - ``"hub_height"``: WSC, corregida a ``hub_height_m``.

    El default es ``measured_50m`` por evidencia, no por comodidad: reproduce el CF eolico
    de 40% que reporta el paper (§3.1) con un desvio de 1%, mientras que corregir a 105 m
    lo lleva a 56%. El paper declara buje a 105 m, de modo que su modelo debe incluir
    perdidas de estela, curva real de turbina u otros descuentos que no publica.
    """

    turbine_rated_mw: float = 5.0
    cut_in_ms: float = 2.5
    rated_ms: float = 10.5
    cut_out_ms: float = 25.0
    hub_height_m: float = 105.0          # Tabla 3 del paper
    wind_speed_source: str = "measured_50m"

    def __post_init__(self):
        if self.turbine_rated_mw <= 0:
            raise ConfigError("turbine_rated_mw debe ser > 0")
        if not self.cut_in_ms < self.rated_ms < self.cut_out_ms:
            raise ConfigError(
                "la curva de potencia exige cut_in < rated < cut_out, se recibio "
                "{} / {} / {}".format(self.cut_in_ms, self.rated_ms, self.cut_out_ms)
            )
        if self.wind_speed_source not in ("measured_50m", "hub_height"):
            raise ConfigError(
                "wind_speed_source debe ser 'measured_50m' o 'hub_height': "
                + str(self.wind_speed_source)
            )


@dataclass(frozen=True)
class PVConfig:
    """Modelo fotovoltaico horario: GHI, derating y correccion termica tipo NOCT.

    ``dc_ac_ratio`` = 1.2 porque el campo tiene mas modulos que la potencia nominal del
    inversor (Tabla 3 del paper: 96.15 MW DC para 80 MW AC). El exceso se recorta en el
    inversor -- *clipping* -- lo que aumenta la produccion en horas de irradiancia media
    sin superar nunca la capacidad AC.
    """

    dc_ac_ratio: float = 1.2
    temp_coeff_pct_per_c: float = -0.5
    noct_c: float = 47.0
    stc_efficiency_pct: float = 13.0
    derating_factor: float = 0.90

    def __post_init__(self):
        if self.dc_ac_ratio <= 0:
            raise ConfigError("dc_ac_ratio debe ser > 0")
        if self.temp_coeff_pct_per_c > 0:
            raise ConfigError("temp_coeff_pct_per_c debe ser <= 0")
        if not 0.0 < self.derating_factor <= 1.0:
            raise ConfigError("derating_factor debe estar en (0, 1]")


@dataclass(frozen=True)
class BatteryConfig:
    """Bateria: eficiencia, dimensionamiento base y malla discreta de la extension.

    En el **modelo base** la potencia es un porcentaje fijo de la del electrolizador,
    redondeado hacia arriba al tamano de pack disponible. Los tres casos publicados lo
    confirman: 0.30*95 = 28.5 -> B30, 0.30*80 = 24 -> B25, 0.30*85 = 25.5 -> B27.5, de
    donde sale el paso de 2.5 MW.

    En la **extension discreta** la potencia y la duracion son variables de decision.
    """

    roundtrip_efficiency: float = 0.90

    # Modelo base
    power_ratio_to_electrolyzer: float = 0.30
    power_rounding_step_mw: float = 2.5
    duration_h: float = 1.0

    # Extension discreta
    power_step_mw: float = 5.0
    power_min_mw: float = 0.0
    power_max_mw: float = 50.0
    duration_candidates_h: Tuple[float, ...] = (1.0, 2.0, 4.0)

    def __post_init__(self):
        if not 0.0 < self.roundtrip_efficiency <= 1.0:
            raise ConfigError("roundtrip_efficiency debe estar en (0, 1]")
        if self.power_rounding_step_mw <= 0 or self.power_step_mw <= 0:
            raise ConfigError("los pasos de potencia deben ser > 0")
        if self.power_min_mw > self.power_max_mw:
            raise ConfigError("power_min_mw > power_max_mw")
        if not self.duration_candidates_h:
            raise ConfigError("duration_candidates_h no puede estar vacio")
        if any(d <= 0 for d in self.duration_candidates_h):
            raise ConfigError("duration_candidates_h debe contener valores > 0")
        # Tupla para que la dataclass congelada siga siendo hasheable.
        object.__setattr__(
            self, "duration_candidates_h",
            tuple(float(d) for d in self.duration_candidates_h),
        )

    @property
    def eta_charge(self):
        """Eficiencia de carga, repartiendo el roundtrip simetricamente."""
        return self.roundtrip_efficiency ** 0.5

    @property
    def eta_discharge(self):
        """Eficiencia de descarga, repartiendo el roundtrip simetricamente."""
        return self.roundtrip_efficiency ** 0.5


@dataclass(frozen=True)
class ElectrolyzerConfig:
    """Electrolizador: eficiencia, carga minima y modularidad discreta."""

    efficiency: float = 0.75
    h2_hhv_kwh_per_kg: float = 39.4
    min_load_ratio: float = 0.30

    # Extension discreta: el electrolizador se compra en unidades enteras.
    unit_mw: float = 5.0
    min_units: int = 10   # 50 MW
    max_units: int = 20   # 100 MW

    # Tanque de hidrogeno (Tabla 3 del paper: 1e4 kg). Entra al NPC cotizado por kg;
    # el paper no lo dimensiona dentro del barrido W/P/E/B, asi que es fijo.
    hydrogen_storage_kg: float = 10_000.0

    def __post_init__(self):
        if not 0.0 < self.efficiency <= 1.0:
            raise ConfigError("efficiency debe estar en (0, 1]")
        if not 0.0 <= self.min_load_ratio <= 1.0:
            raise ConfigError("min_load_ratio debe estar en [0, 1]")
        if self.unit_mw <= 0:
            raise ConfigError("unit_mw debe ser > 0")
        if self.min_units > self.max_units:
            raise ConfigError("min_units > max_units")

    @property
    def specific_consumption_kwh_per_kg(self):
        """Consumo especifico de electricidad por kg de H2.

        Derivado, no configurable: con los valores del paper resulta
        39.4 / 0.75 = 52.53 kWh/kg. Que sea una propiedad garantiza que el modelo base
        y el extendido usen exactamente la misma conversion.
        """
        return self.h2_hhv_kwh_per_kg / self.efficiency

    @property
    def min_capacity_mw(self):
        return self.min_units * self.unit_mw

    @property
    def max_capacity_mw(self):
        return self.max_units * self.unit_mw


@dataclass(frozen=True)
class ConverterConfig:
    """Eficiencias de conversion y consumo parasito de la planta.

    El balance de potencia del paper (Eq. 6) contabiliza perdidas de inversor,
    rectificador y bateria mas la carga de planta. La Tabla 3 fija los convertidores
    en 95%. ``plant_load_ratio`` queda en 0.0 porque el paper no publica su valor:
    es un supuesto explicito, no un dato.
    """

    inverter_efficiency: float = 0.95        # PV DC -> AC
    rectifier_efficiency: float = 0.95       # AC -> DC del electrolizador
    bidirectional_efficiency: float = 0.95   # bateria AC <-> DC
    plant_load_ratio: float = 0.0

    def __post_init__(self):
        for name in ("inverter_efficiency", "rectifier_efficiency",
                     "bidirectional_efficiency"):
            if not 0.0 < getattr(self, name) <= 1.0:
                raise ConfigError(name + " debe estar en (0, 1]")
        if not 0.0 <= self.plant_load_ratio < 1.0:
            raise ConfigError("plant_load_ratio debe estar en [0, 1)")


@dataclass(frozen=True)
class EconomicsConfig:
    """Horizonte, descuento y criterio de energia util para el LCOE.

    ``lcoe_energy_basis`` decide el denominador del LCOE: ``electrolyzer_plus_grid``
    cuenta toda la energia util (la que va al electrolizador mas la exportada),
    ``electrolyzer_only`` cuenta solo la del electrolizador. El paper usa la primera.
    """

    project_lifetime_years: int = 25
    real_discount_rate: float = 0.0435
    lcoe_energy_basis: str = "electrolyzer_plus_grid"

    def __post_init__(self):
        if self.project_lifetime_years < 1:
            raise ConfigError("project_lifetime_years debe ser >= 1")
        if self.real_discount_rate <= -1.0:
            raise ConfigError("real_discount_rate debe ser > -1")
        if self.lcoe_energy_basis not in ("electrolyzer_plus_grid", "electrolyzer_only"):
            raise ConfigError(
                "lcoe_energy_basis invalido: " + str(self.lcoe_energy_basis)
            )


@dataclass(frozen=True)
class ConstraintsConfig:
    """Restricciones de diseno del paper."""

    total_generation_capacity_mw: float = 200.0   # Wind + PV, fijo
    agsr_max: float = 0.20                        # ventas a red / generacion renovable
    electrolyzer_ratio_max: float = 0.50          # E <= 50% de (Wind + PV)
    grid_export_limit_mw: Optional[float] = None  # None -> exportacion ilimitada

    def __post_init__(self):
        if self.total_generation_capacity_mw <= 0:
            raise ConfigError("total_generation_capacity_mw debe ser > 0")
        if not 0.0 <= self.agsr_max <= 1.0:
            raise ConfigError("agsr_max debe estar en [0, 1]")
        if not 0.0 < self.electrolyzer_ratio_max <= 1.0:
            raise ConfigError("electrolyzer_ratio_max debe estar en (0, 1]")

    @property
    def electrolyzer_max_mw(self):
        return self.electrolyzer_ratio_max * self.total_generation_capacity_mw


@dataclass(frozen=True)
class SearchConfig:
    """Resolucion del barrido exhaustivo de la instancia base."""

    wind_step_mw: float = 10.0
    electrolyzer_step_mw: float = 5.0

    def __post_init__(self):
        if self.wind_step_mw <= 0 or self.electrolyzer_step_mw <= 0:
            raise ConfigError("los pasos del barrido deben ser > 0")


@dataclass(frozen=True)
class ComponentCost:
    """Costos de un componente, con base de potencia (CNY/kW) y de energia (CNY/kWh).

    La base de energia existe sobre todo para la bateria: si la duracion es variable de
    decision pero no tiene costo por kWh, alargarla es gratis y el optimizador la lleva
    al maximo sin que ese resultado signifique nada.
    """

    capex_cny_per_kw: float = 0.0
    replacement_cny_per_kw: float = 0.0
    om_cny_per_kw_year: float = 0.0
    capex_cny_per_kwh: float = 0.0
    replacement_cny_per_kwh: float = 0.0
    om_cny_per_kwh_year: float = 0.0
    life_years: int = 25

    def __post_init__(self):
        for f in fields(self):
            if f.name != "life_years" and getattr(self, f.name) < 0:
                raise ConfigError(f.name + " debe ser >= 0")
        if self.life_years < 1:
            raise ConfigError("life_years debe ser >= 1")

    @property
    def has_energy_cost(self):
        return (self.capex_cny_per_kwh > 0
                or self.replacement_cny_per_kwh > 0
                or self.om_cny_per_kwh_year > 0)


@dataclass(frozen=True)
class CostConfig:
    """Costos del paper (Tabla 4), en CNY."""

    wind: ComponentCost = field(default_factory=lambda: ComponentCost(
        capex_cny_per_kw=5917.0, replacement_cny_per_kw=0.0,
        om_cny_per_kw_year=40.2, life_years=25))
    pv: ComponentCost = field(default_factory=lambda: ComponentCost(
        capex_cny_per_kw=4633.0, replacement_cny_per_kw=0.0,
        om_cny_per_kw_year=17.6, life_years=25))
    electrolyzer: ComponentCost = field(default_factory=lambda: ComponentCost(
        capex_cny_per_kw=6964.0, replacement_cny_per_kw=5969.14,
        om_cny_per_kw_year=208.92, life_years=15))
    battery: ComponentCost = field(default_factory=lambda: ComponentCost(
        capex_cny_per_kw=2549.0, replacement_cny_per_kw=500.0,
        om_cny_per_kw_year=10.0, life_years=10))

    hydrogen_storage_capex_cny_per_kg: float = 6611.77
    hydrogen_storage_om_cny_per_kg_year: float = 141.68
    hydrogen_storage_life_years: int = 25

    def __post_init__(self):
        if self.hydrogen_storage_capex_cny_per_kg < 0:
            raise ConfigError("hydrogen_storage_capex_cny_per_kg debe ser >= 0")

## 3. Configuracion de las metaheuristicas

Esta es la seccion que hace que la comparativa sea auditable: **cada hiperparametro de
cada algoritmo esta declarado aqui**, fuera del codigo del operador de busqueda. Ningun
algoritmo tiene numeros magicos escondidos en su bucle.

Los valores son los **canonicos de la literatura**, no un ajuste fino a este problema:

- **PSO**: coeficientes de constriccion de Clerc & Kennedy (2002), `chi = 0.729`,
  `c1 = c2 = 1.49445`.
- **GA**: real-coded estandar -- torneo `k = 3`, cruce BLX-alpha, mutacion gaussiana con
  `sigma` igual al 10% del rango de cada variable, elitismo de 2.
- **SHO**: `h_max = 5.0`, el valor de Dhiman & Kumar (2017).
- **WOA**: `b = 1.0` (constante de la espiral), el valor de Mirjalili & Lewis (2016).

Esto importa para leer los resultados: la comparativa **no** debe interpretarse como un
torneo con un favorito afinado. Si se ajustaran los hiperparametros de un algoritmo y no
los de los demas, el ganador seria el que recibio mas atencion, no el mejor metodo.

`population` e `iterations` fijan el **presupuesto de evaluaciones** (`population x
iterations`), identico para los 6 algoritmos. Es la condicion para que la comparacion sea
justa: todos ven exactamente el mismo numero de disenos.

La **penalizacion** de las soluciones infactibles tambien vive aqui:
`penalty_infeasible + penalty_agsr_weight * exceso_de_AGSR`. Es proporcional a cuanto se
viola la restriccion, no un valor plano, para que el optimizador tenga gradiente hacia la
region factible en vez de un muro.

In [ ]:
@dataclass(frozen=True)
class PSOConfig:
    """Hiperparametros del enjambre de particulas.

    Formulacion de **constriccion** de Clerc-Kennedy, no la de peso de inercia clasico:
    ``inertia`` multiplica toda la suma, no solo la velocidad previa, y es lo que
    garantiza convergencia sin necesidad de decaerla con las iteraciones.

        v = chi * (v + c1*r1*(pbest - x) + c2*r2*(gbest - x))
    """

    inertia: float = 0.729              # chi
    cognitive: float = 1.49445          # c1
    social: float = 1.49445             # c2
    velocity_clamp_ratio: float = 0.20  # |v| <= 20% del rango de cada variable

    def __post_init__(self):
        if self.inertia < 0 or self.cognitive < 0 or self.social < 0:
            raise ConfigError("los coeficientes de PSO deben ser >= 0")
        if not 0.0 < self.velocity_clamp_ratio <= 1.0:
            raise ConfigError("velocity_clamp_ratio debe estar en (0, 1]")


@dataclass(frozen=True)
class GAConfig:
    """Hiperparametros del algoritmo genetico de codificacion real."""

    crossover_rate: float = 0.90        # probabilidad de cruzar en vez de clonar
    mutation_rate: float = 0.10         # probabilidad por gen
    mutation_sigma_ratio: float = 0.10  # sigma = 10% del rango de cada variable
    tournament_size: int = 3            # k del torneo
    elite_count: int = 2                # individuos que pasan intactos
    blx_alpha: float = 0.5              # expansion del intervalo en el cruce BLX

    def __post_init__(self):
        if not 0.0 <= self.crossover_rate <= 1.0:
            raise ConfigError("crossover_rate debe estar en [0, 1]")
        if not 0.0 <= self.mutation_rate <= 1.0:
            raise ConfigError("mutation_rate debe estar en [0, 1]")
        if self.mutation_sigma_ratio <= 0:
            raise ConfigError("mutation_sigma_ratio debe ser > 0")
        if self.tournament_size < 2:
            raise ConfigError("tournament_size debe ser >= 2")
        if self.elite_count < 0:
            raise ConfigError("elite_count debe ser >= 0")
        if self.blx_alpha < 0:
            raise ConfigError("blx_alpha debe ser >= 0")


@dataclass(frozen=True)
class SHOConfig:
    """Hiperparametros del Spotted Hyena Optimizer (Dhiman & Kumar, 2017).

    ``h_max`` es el unico hiperparametro de la literatura: el coeficiente ``h`` decae
    linealmente de ``h_max`` a 0 (en GWO el analogo ``a`` decae de 2 a 0). Un rango mas
    ancho da mas exploracion inicial con la misma estructura de encierro.
    """

    h_max: float = 5.0

    def __post_init__(self):
        if self.h_max <= 0:
            raise ConfigError("h_max debe ser > 0")


@dataclass(frozen=True)
class WOAConfig:
    """Hiperparametros del Whale Optimization Algorithm (Mirjalili & Lewis, 2016).

    ``spiral_constant`` (``b`` en el paper) fija la forma de la espiral logaritmica de la
    fase de ataque por burbujas; 1.0 es el valor que usan los autores.
    """

    spiral_constant: float = 1.0

    def __post_init__(self):
        if self.spiral_constant <= 0:
            raise ConfigError("spiral_constant debe ser > 0")


@dataclass(frozen=True)
class MetaheuristicConfig:
    """Presupuesto, penalizacion y reproducibilidad de los optimizadores."""

    population: int = 20
    iterations: int = 30
    seed: int = 42
    penalty_infeasible: float = 1.0e6
    penalty_agsr_weight: float = 1.0e3
    pso: PSOConfig = field(default_factory=PSOConfig)
    ga: GAConfig = field(default_factory=GAConfig)
    sho: SHOConfig = field(default_factory=SHOConfig)
    woa: WOAConfig = field(default_factory=WOAConfig)

    def __post_init__(self):
        if self.population < 1:
            raise ConfigError("population debe ser >= 1")
        if self.iterations < 1:
            raise ConfigError("iterations debe ser >= 1")
        if self.penalty_infeasible < 0 or self.penalty_agsr_weight < 0:
            raise ConfigError("las penalizaciones deben ser >= 0")
        if self.ga.elite_count > self.population:
            raise ConfigError("ga.elite_count ({}) no puede superar population ({})".format(
                self.ga.elite_count, self.population))
        if self.ga.tournament_size > self.population:
            raise ConfigError("ga.tournament_size ({}) no puede superar population ({})".format(
                self.ga.tournament_size, self.population))

    @property
    def evaluation_budget(self):
        """Evaluaciones del objetivo por corrida. Identico para los 6 algoritmos."""
        return self.population * self.iterations

## 4. El escenario resuelto

Se arma el `ScenarioConfig` completo con **una sola desviacion** respecto de los valores
del paper, declarada explicitamente:

**Reparto del costo de la bateria entre potencia y energia.** El paper solo cotiza la
bateria por kW (2549 / 500 / 10 CNY/kW de capex / reemplazo / O&M) porque sus tres casos
publicados son todos de 1 h, donde potencia y energia son numericamente iguales y no se
pueden separar. Para que la duracion de la bateria tenga sentido economico como variable
de decision, este escenario reparte esos costos **30% potencia / 70% energia**:

| | por kW | por kWh | suma a 1 h |
|---|---|---|---|
| capex | 764.70 | 1784.30 | **2549.0** |
| reemplazo | 150.0 | 350.0 | **500.0** |
| O&M anual | 3.0 | 7.0 | **10.0** |

A 1 h la suma da exactamente los valores del paper -- el reparto es **NPC-neutro**, de
modo que la validacion contra el paper (seccion 10) y el barrido base (seccion 11), que
usan bateria de 1 h, no se mueven. A mas horas la bateria cuesta mas, que es lo que hace
que el optimo en duracion signifique algo. **Es un supuesto explicito, no un dato del
paper.**

In [ ]:
@dataclass(frozen=True)
class ScenarioConfig:
    """Escenario completo: todo lo necesario para reproducir una corrida."""

    name: str = "metaheuristicas"
    description: str = ""
    site: SiteConfig = field(default_factory=SiteConfig)
    wind: WindConfig = field(default_factory=WindConfig)
    pv: PVConfig = field(default_factory=PVConfig)
    battery: BatteryConfig = field(default_factory=BatteryConfig)
    electrolyzer: ElectrolyzerConfig = field(default_factory=ElectrolyzerConfig)
    converter: ConverterConfig = field(default_factory=ConverterConfig)
    economics: EconomicsConfig = field(default_factory=EconomicsConfig)
    constraints: ConstraintsConfig = field(default_factory=ConstraintsConfig)
    search: SearchConfig = field(default_factory=SearchConfig)
    metaheuristic: MetaheuristicConfig = field(default_factory=MetaheuristicConfig)
    costs: CostConfig = field(default_factory=CostConfig)

    def __post_init__(self):
        # Coherencia entre bloques: la malla discreta del electrolizador no puede
        # violar la restriccion E <= ratio_max * (Wind + PV).
        max_discrete = self.electrolyzer.max_capacity_mw
        allowed = self.constraints.electrolyzer_max_mw
        if max_discrete > allowed + 1e-9:
            raise ConfigError(
                "la malla discreta permite hasta {:.1f} MW de electrolizador, pero la "
                "restriccion admite como maximo {:.1f} MW".format(max_discrete, allowed)
            )

    def replace(self, **changes):
        """Copia con cambios, revalidando. Util para los barridos de sensibilidad."""
        return replace(self, **changes)


# Reparto NPC-neutro 30/70 de los costos de bateria del paper (ver celda anterior).
BATERIA_REPARTIDA = ComponentCost(
    capex_cny_per_kw=764.70,        # 2549 * 0.30
    capex_cny_per_kwh=1784.30,      # 2549 * 0.70
    replacement_cny_per_kw=150.0,   # 500 * 0.30
    replacement_cny_per_kwh=350.0,  # 500 * 0.70
    om_cny_per_kw_year=3.0,         # 10 * 0.30
    om_cny_per_kwh_year=7.0,        # 10 * 0.70
    life_years=10,
)

ESCENARIO = ScenarioConfig(
    name="metaheuristicas",
    description=(
        "Extension discreta de Li et al. (2024): electrolizador modular, potencia y "
        "duracion de bateria como variables de decision."
    ),
    costs=CostConfig(battery=BATERIA_REPARTIDA),
    metaheuristic=MetaheuristicConfig(
        population=PRESUPUESTO["population"],
        iterations=PRESUPUESTO["iterations"],
        seed=42,
    ),
)

SEMILLAS = tuple(range(ESCENARIO.metaheuristic.seed,
                       ESCENARIO.metaheuristic.seed + PRESUPUESTO["n_seeds"]))


def _tabla_config(obj, prefix=""):
    """Aplana una jerarquia de dataclasses a filas (parametro, valor)."""
    rows = []
    for f in fields(obj):
        value = getattr(obj, f.name)
        name = prefix + f.name
        if is_dataclass(value):
            rows.extend(_tabla_config(value, name + "."))
        else:
            rows.append({"parametro": name, "valor": value})
    return rows


print("Escenario :", ESCENARIO.name)
print("Semillas  :", SEMILLAS[0], "..", SEMILLAS[-1], "({} corridas por algoritmo)".format(len(SEMILLAS)))
print("Presupuesto por corrida:", ESCENARIO.metaheuristic.evaluation_budget, "evaluaciones")
print("Consumo especifico del electrolizador: {:.2f} kWh/kg".format(
    ESCENARIO.electrolyzer.specific_consumption_kwh_per_kg))
print("Electrolizador maximo admitido por la restriccion: {:.0f} MW".format(
    ESCENARIO.constraints.electrolyzer_max_mw))

display(pd.DataFrame(_tabla_config(ESCENARIO)))

## 5. El dato horario

`src/hourly_2008.csv` es el unico insumo externo. Cada fila es una hora del ano 2008 en
Damao Banner, con cuatro variables provenientes de NASA POWER:

| columna | significado |
|---|---|
| `ws50m` | velocidad de viento medida a 50 m [m/s] |
| `wsc_ms` | velocidad corregida a la altura de buje (105 m) [m/s] |
| `ghi_kwh_m2` | irradiancia global horizontal [kWh/m2 por hora] |
| `t2m_c` | temperatura a 2 m [C] |

Son **8784 filas**, no 8760, porque 2008 fue bisiesto: 366 dias x 24 horas. El paper
identifica 2008 como el ano meteorologico tipico del sitio.

El escenario usa `ws50m` (ver la nota en `WindConfig`); `wsc_ms` queda disponible para
poder medir por separado el efecto de la extrapolacion a altura de buje.

In [ ]:
hourly = pd.read_csv(HOURLY_PATH, parse_dates=["timestamp"])

print("Filas horarias:", len(hourly), "({} dias)".format(len(hourly) // 24))
print("Columnas      :", hourly.columns.tolist())
print("Periodo       :", hourly["timestamp"].min(), "->", hourly["timestamp"].max())
print("Viento medio a 50 m : {:.2f} m/s".format(hourly["ws50m"].mean()))
print("Viento medio a buje : {:.2f} m/s".format(hourly["wsc_ms"].mean()))
print("GHI anual           : {:.0f} kWh/m2".format(hourly["ghi_kwh_m2"].sum()))

display(hourly.head())
display(hourly.describe())

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
muestra = hourly.iloc[:24 * 14]

axes[0].plot(muestra["timestamp"], muestra["ws50m"], color="#2a78d6", lw=1.1)
axes[0].set_ylabel("Viento 50 m\n[m/s]")
axes[1].plot(muestra["timestamp"], muestra["ghi_kwh_m2"], color="#eda100", lw=1.1)
axes[1].set_ylabel("GHI\n[kWh/m2]")
axes[2].plot(muestra["timestamp"], muestra["t2m_c"], color="#e87ba4", lw=1.1)
axes[2].set_ylabel("Temperatura\n[C]")
axes[2].set_xlabel("Fecha")

fig.suptitle("Primeras dos semanas del ano meteorologico 2008 -- Damao Banner")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 6. Modelos de generacion

### Eolica

Curva de potencia cubica simplificada, con tres regimenes:

$$P(v) = \begin{cases}
0 & v < v_{ci} \ \text{o} \ v \ge v_{co} \\[4pt]
P_r \dfrac{v^3 - v_{ci}^3}{v_r^3 - v_{ci}^3} & v_{ci} \le v < v_r \\[8pt]
P_r & v_r \le v < v_{co}
\end{cases}$$

El parque son `round(capacidad / 5 MW)` turbinas identicas, y la salida se satura a la
capacidad nominal declarada para que el redondeo hacia arriba no genere mas potencia de
la instalada.

### Fotovoltaica

La GHI horaria en kWh/m2 se interpreta como irradiancia media en kW/m2, de modo que su
valor coincide numericamente con la produccion relativa antes de perdidas. Sobre eso se
aplican el derating (0.90) y la correccion termica tipo NOCT:

$$T_{cell} = T_{amb} + \frac{NOCT - 20}{0.8}\,G \qquad
f_T = 1 + \gamma\,(T_{cell} - 25)$$

La potencia DC se calcula sobre `capacidad_AC x 1.2` y luego se recorta en el inversor,
que es lo que produce el *clipping*.

### Cache de perfiles

`GenerationProfileCache` memoiza los perfiles por capacidad. Sin el, el barrido de la
seccion 11 recalcularia el perfil eolico 420 veces cuando alcanzan 21, y la optimizacion
lo recalcularia en cada una de las 108.000 evaluaciones. **El resultado numerico es
identico**; lo unico que cambia es el tiempo.

In [ ]:
def wind_turbine_power_mw(wind_speed_ms, rated_mw, cut_in_ms, rated_ms, cut_out_ms):
    """Potencia de UNA turbina, con curva cubica simplificada."""
    ws = np.asarray(wind_speed_ms, dtype=float)
    p = np.zeros_like(ws, dtype=float)

    ramp = (ws >= cut_in_ms) & (ws < rated_ms)
    plateau = (ws >= rated_ms) & (ws < cut_out_ms)

    p[ramp] = rated_mw * ((ws[ramp] ** 3 - cut_in_ms ** 3)
                          / (rated_ms ** 3 - cut_in_ms ** 3))
    p[plateau] = rated_mw
    return np.clip(p, 0.0, rated_mw)


def aggregate_wind_power_mw(wind_speed_ms, wind_capacity_mw, config):
    """Potencia del parque completo, saturada a la capacidad nominal."""
    n_turbines = max(int(round(wind_capacity_mw / config.turbine_rated_mw)), 0)
    single = wind_turbine_power_mw(
        wind_speed_ms,
        rated_mw=config.turbine_rated_mw,
        cut_in_ms=config.cut_in_ms,
        rated_ms=config.rated_ms,
        cut_out_ms=config.cut_out_ms,
    )
    return np.minimum(single * n_turbines, wind_capacity_mw)


def cell_temperature_c(ghi_kwh_m2, ambient_c, config):
    """Temperatura de celda, aproximacion tipo NOCT."""
    ghi = np.asarray(ghi_kwh_m2, dtype=float)
    t_amb = np.asarray(ambient_c, dtype=float)
    return t_amb + (config.noct_c - 20.0) / 0.8 * ghi


def pv_power_mw(ghi_kwh_m2, ambient_c, pv_ac_capacity_mw, config,
                inverter_efficiency=1.0):
    """Potencia AC del campo fotovoltaico, con clipping en el inversor."""
    ghi = np.asarray(ghi_kwh_m2, dtype=float)
    t_cell = cell_temperature_c(ghi, ambient_c, config)

    gamma = config.temp_coeff_pct_per_c / 100.0
    temp_factor = 1.0 + gamma * (t_cell - 25.0)

    relative = ghi * config.derating_factor * np.maximum(temp_factor, 0.0)

    dc_capacity_mw = pv_ac_capacity_mw * config.dc_ac_ratio
    power_dc = dc_capacity_mw * np.maximum(relative, 0.0)
    power_ac = power_dc * inverter_efficiency
    return np.minimum(power_ac, pv_ac_capacity_mw)


HUB_WIND_COLUMN = "wsc_ms"
MEASURED_WIND_COLUMN = "ws50m"
_QUANTUM_MW = 1e-6   # cuantizacion para que 190.0 y 190.0000000001 compartan cache


class GenerationProfileCache:
    """Perfiles horarios de generacion para un ano meteorologico dado."""

    def __init__(self, hourly, config):
        missing = [c for c in ("ghi_kwh_m2", "t2m_c") if c not in hourly.columns]
        if missing:
            raise ValueError("faltan columnas en los datos horarios: {}".format(missing))

        if config.wind.wind_speed_source == "hub_height":
            if HUB_WIND_COLUMN not in hourly.columns:
                raise ValueError(
                    "el escenario pide viento a altura de buje pero falta la columna "
                    + HUB_WIND_COLUMN
                )
            wind_column = HUB_WIND_COLUMN
        else:
            if MEASURED_WIND_COLUMN not in hourly.columns:
                raise ValueError("falta la columna " + MEASURED_WIND_COLUMN)
            wind_column = MEASURED_WIND_COLUMN

        self._config = config
        self._n_hours = len(hourly)
        self._wind_column = wind_column
        self._wind_speed = np.ascontiguousarray(hourly[wind_column].to_numpy(dtype=float))
        self._ghi = np.ascontiguousarray(hourly["ghi_kwh_m2"].to_numpy(dtype=float))
        self._temperature = np.ascontiguousarray(hourly["t2m_c"].to_numpy(dtype=float))

        self._wind_cache = {}
        self._pv_cache = {}
        self._total_cache = {}
        self.hits = 0
        self.misses = 0

    @property
    def n_hours(self):
        return self._n_hours

    @property
    def wind_column(self):
        """Columna de viento efectivamente usada: util para auditar una corrida."""
        return self._wind_column

    @staticmethod
    def _key(capacity_mw):
        return int(round(capacity_mw / _QUANTUM_MW))

    def wind(self, wind_mw):
        key = self._key(wind_mw)
        profile = self._wind_cache.get(key)
        if profile is None:
            profile = aggregate_wind_power_mw(self._wind_speed, wind_mw, self._config.wind)
            self._wind_cache[key] = profile
        return profile

    def pv(self, pv_mw):
        key = self._key(pv_mw)
        profile = self._pv_cache.get(key)
        if profile is None:
            profile = pv_power_mw(
                self._ghi, self._temperature, pv_mw, self._config.pv,
                inverter_efficiency=self._config.converter.inverter_efficiency,
            )
            self._pv_cache[key] = profile
        return profile

    def total(self, wind_mw, pv_mw):
        """Generacion renovable agregada, lista para el despacho."""
        key = (self._key(wind_mw), self._key(pv_mw))
        profile = self._total_cache.get(key)
        if profile is None:
            self.misses += 1
            profile = np.ascontiguousarray(self.wind(wind_mw) + self.pv(pv_mw))
            self._total_cache[key] = profile
        else:
            self.hits += 1
        return profile

    def stats(self):
        return {"hits": self.hits, "misses": self.misses,
                "wind_profiles": len(self._wind_cache),
                "pv_profiles": len(self._pv_cache)}


def as_profile_cache(hourly, config):
    """Acepta indistintamente un DataFrame horario o un cache ya construido."""
    if isinstance(hourly, GenerationProfileCache):
        return hourly
    return GenerationProfileCache(hourly, config)

In [ ]:
# Curva de potencia de la turbina y factores de capacidad del recurso.
cache_demo = as_profile_cache(hourly, ESCENARIO)
horas = cache_demo.n_hours

cf_wind = float(cache_demo.wind(100.0).sum()) / (100.0 * horas)
cf_pv = float(cache_demo.pv(100.0).sum()) / (100.0 * horas)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

v = np.linspace(0, 28, 400)
axes[0].plot(v, wind_turbine_power_mw(v, 5.0, 2.5, 10.5, 25.0), color="#2a78d6", lw=2)
axes[0].axvline(2.5, ls=":", color="grey"); axes[0].axvline(10.5, ls=":", color="grey")
axes[0].axvline(25.0, ls=":", color="grey")
axes[0].set_xlabel("Velocidad de viento [m/s]")
axes[0].set_ylabel("Potencia por turbina [MW]")
axes[0].set_title("Curva de potencia (cut-in 2.5, nominal 10.5, cut-out 25)")

dia = slice(24 * 180, 24 * 183)
axes[1].plot(hourly["timestamp"].iloc[dia], cache_demo.wind(100.0)[dia],
             label="Eolica 100 MW", color="#2a78d6")
axes[1].plot(hourly["timestamp"].iloc[dia], cache_demo.pv(100.0)[dia],
             label="PV 100 MW", color="#eda100")
axes[1].set_ylabel("Potencia [MW]")
axes[1].set_title("Tres dias de verano, parques de referencia")
axes[1].legend()

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print("Columna de viento usada : {}".format(cache_demo.wind_column))
print("CF eolico  (100 MW): {:.4f}   (paper: 0.40)".format(cf_wind))
print("CF fotovoltaico    : {:.4f}   (paper: 0.19)".format(cf_pv))

## 7. Modelo economico

El criterio es el **NPC** (*Net Present Cost*) y su anualizacion via el factor de
recuperacion de capital:

$$CRF(i, N) = \frac{i\,(1+i)^N}{(1+i)^N - 1}$$

Para cada componente:

$$NPC = C_{capex} + \sum_{y \in \text{reemplazos}} \frac{C_{rep}}{(1+i)^y}
      + \sum_{y=1}^{N} \frac{C_{O\&M}}{(1+i)^y} - S$$

**Valor residual $S$** (metodo lineal de HOMER): si el ultimo reemplazo ocurre antes del
fin del horizonte, al componente le queda vida util sin usar y ese remanente se valora en
proporcion y se descuenta desde el fin del proyecto:

$$S = C_{rep}\,\frac{\text{vida restante}}{\text{vida del componente}}\,\frac{1}{(1+i)^N}$$

Con los datos del paper, el electrolizador (vida 15 anos, reemplazo en el ano 15) llega
al ano 25 con 5 de 15 anos sin usar. La eolica y la PV, cuya vida coincide con el
horizonte, no tienen residual. Ignorar el residual sobreestima el NPC del electrolizador,
que es el componente mas caro despues de la generacion.

Sobre esa base se calculan los dos costos nivelados:

$$LCOE = \frac{NPC \cdot CRF}{E_{util}} \qquad LCOH = \frac{NPC \cdot CRF}{m_{H_2}}$$

In [ ]:
def crf(discount_rate, years):
    """Factor de recuperacion de capital.

    Con tasa cero degenera en 1/n, que es el limite analitico y evita dividir por cero.
    """
    if years < 1:
        raise ValueError("years debe ser >= 1")
    if discount_rate == 0.0:
        return 1.0 / years
    growth = (1.0 + discount_rate) ** years
    return (discount_rate * growth) / (growth - 1.0)


def replacement_years(life_years, project_years):
    """Anos en que toca reemplazar dentro del horizonte.

    Un componente de vida 15 en un proyecto de 25 se reemplaza solo en el ano 15:
    el del ano 30 cae fuera del horizonte.
    """
    if life_years < 1:
        raise ValueError("life_years debe ser >= 1")
    return list(range(life_years, project_years, life_years))


def present_value_of_replacements(quantity, cost_per_unit, life_years,
                                  project_years, discount_rate):
    """Valor presente de los reemplazos periodicos."""
    if quantity <= 0 or cost_per_unit <= 0:
        return 0.0
    return sum((quantity * cost_per_unit) / ((1.0 + discount_rate) ** year)
               for year in replacement_years(life_years, project_years))


def present_value_of_om(quantity, om_per_unit_year, project_years, discount_rate):
    """Valor presente de la operacion y mantenimiento anual."""
    if quantity <= 0 or om_per_unit_year <= 0:
        return 0.0
    return sum((quantity * om_per_unit_year) / ((1.0 + discount_rate) ** year)
               for year in range(1, project_years + 1))


def salvage_value(quantity, replacement_cost_per_unit, life_years,
                  project_years, discount_rate):
    """Valor residual al final del horizonte, metodo lineal de HOMER."""
    if quantity <= 0 or replacement_cost_per_unit <= 0:
        return 0.0

    replacements = replacement_years(life_years, project_years)
    if not replacements:
        return 0.0

    last_replacement = replacements[-1]
    remaining_life = life_years - (project_years - last_replacement)
    if remaining_life <= 0:
        return 0.0

    value = quantity * replacement_cost_per_unit * (remaining_life / life_years)
    return value / ((1.0 + discount_rate) ** project_years)


def component_npc(capacity_kw, capacity_kwh, cost, economics):
    """NPC de un componente: capex + reemplazos + O&M - residual.

    Suma la base de potencia y la de energia. Para los componentes del paper la base
    de energia es 0.0, asi que solo contribuye la de potencia; para la bateria de este
    escenario contribuyen las dos.
    """
    rate = economics.real_discount_rate
    horizon = economics.project_lifetime_years

    total = 0.0
    for quantity, capex, replacement, om in (
        (capacity_kw, cost.capex_cny_per_kw, cost.replacement_cny_per_kw,
         cost.om_cny_per_kw_year),
        (capacity_kwh, cost.capex_cny_per_kwh, cost.replacement_cny_per_kwh,
         cost.om_cny_per_kwh_year),
    ):
        if quantity <= 0:
            continue
        total += quantity * capex
        total += present_value_of_replacements(quantity, replacement, cost.life_years,
                                               horizon, rate)
        total += present_value_of_om(quantity, om, horizon, rate)
        total -= salvage_value(quantity, replacement, cost.life_years, horizon, rate)
    return total


def hydrogen_storage_npc(capacity_kg, costs, economics):
    """NPC del almacenamiento de H2, cotizado por kg (Tabla 4 del paper).

    Su vida util coincide con el horizonte, de modo que no tiene reemplazos ni residual.
    """
    if capacity_kg <= 0:
        return 0.0
    return capacity_kg * costs.hydrogen_storage_capex_cny_per_kg + present_value_of_om(
        capacity_kg, costs.hydrogen_storage_om_cny_per_kg_year,
        economics.project_lifetime_years, economics.real_discount_rate,
    )


def npc_from_capacities(wind_mw, pv_mw, electrolyzer_mw, battery_mw, battery_mwh,
                        costs, economics, hydrogen_storage_kg=0.0):
    """NPC del sistema WPEB completo."""
    kw = 1000.0
    return (
        component_npc(wind_mw * kw, 0.0, costs.wind, economics)
        + component_npc(pv_mw * kw, 0.0, costs.pv, economics)
        + component_npc(electrolyzer_mw * kw, 0.0, costs.electrolyzer, economics)
        + component_npc(battery_mw * kw, battery_mwh * kw, costs.battery, economics)
        + hydrogen_storage_npc(hydrogen_storage_kg, costs, economics)
    )


def levelized_cost(annualized_cost_cny, annual_output):
    """Costo nivelado. Infinito si no hay produccion, para que una configuracion
    esteril nunca gane una comparacion por minimo."""
    if annual_output <= 0:
        return float("inf")
    return annualized_cost_cny / annual_output


print("CRF(4.35%, 25 anos) = {:.6f}".format(
    crf(ESCENARIO.economics.real_discount_rate,
        ESCENARIO.economics.project_lifetime_years)))
print("Reemplazos del electrolizador (vida 15, horizonte 25): anos", replacement_years(15, 25))
print("Reemplazos de la bateria      (vida 10, horizonte 25): anos", replacement_years(10, 25))

## 8. Despacho horario

Este es el nucleo del modelo y el unico bucle secuencial: el estado de carga de la
bateria en cada hora depende de la anterior, de modo que no se puede vectorizar sin
cambiar el modelo.

**Prioridad de uso de la energia en cada hora:**

1. Alimentar el electrolizador (hasta su capacidad nominal).
2. Cargar la bateria con el excedente.
3. Exportar a la red lo que sobre (esto es lo que cuenta el AGSR).
4. Recortar (*curtailment*) lo que exceda el limite de conexion, si lo hay.

**Dos politicas de carga minima.** El electrolizador no puede operar por debajo del 30%
de su capacidad. Que hacer cuando la generacion cae bajo ese umbral define dos modelos
distintos, y aqui viven los dos en el mismo nucleo para que sus resultados sigan siendo
comparables:

| politica | comportamiento | usada por |
|---|---|---|
| `TOP_UP` | El electrolizador toma toda la generacion disponible; si queda bajo el minimo, la bateria intenta completar el deficit. Si aun asi no llega, se apaga y la descarga se revierte. | modelo base (seccion 11) |
| `THRESHOLD` | Si la generacion **por si sola** alcanza el minimo, opera. Si no, la bateria intenta sostener el minimo; si no puede, se apaga sin descargar nada. | extension discreta (seccion 16) |

La diferencia es sutil pero cambia los numeros: `TOP_UP` gasta bateria en horas de
generacion baja, `THRESHOLD` la reserva.

In [ ]:
class MinLoadPolicy(IntEnum):
    """Regla de operacion del electrolizador bajo la carga minima."""

    TOP_UP = 0     # modelo base
    THRESHOLD = 1  # extension discreta


class DispatchResult(NamedTuple):
    """Series horarias resultantes del despacho, en MW (SOC en MWh)."""

    electrolyzer_load: np.ndarray
    grid_sales: np.ndarray
    battery_charge: np.ndarray
    battery_discharge: np.ndarray
    curtailment: np.ndarray
    state_of_charge: np.ndarray


# Cualquier valor negativo del limite de exportacion significa "sin limite".
_NO_GRID_LIMIT = -1.0


def _dispatch_core(generation_mw, electrolyzer_mw, min_load_mw, battery_power_mw,
                   battery_energy_mwh, eta_charge, eta_discharge, policy,
                   grid_export_limit_mw):
    """Bucle secuencial sobre el estado de carga.

    Acumula en listas de Python en vez de arrays de NumPy: evita crear un escalar de
    NumPy por cada una de las 8784 horas, que es lo que domina el costo de esta ruta.
    """
    n = len(generation_mw)

    electrolyzer_load = []
    grid_sales = []
    battery_charge = []
    battery_discharge = []
    curtailment = []
    state_of_charge = []

    soc = 0.0

    for t in range(n):
        generation = generation_mw[t]
        load = 0.0
        discharge = 0.0
        charge = 0.0
        exported = 0.0
        curtailed = 0.0

        if policy == 0:
            # TOP_UP: consumir todo lo posible y completar con bateria si el
            # resultado cae bajo la carga minima.
            load = generation if generation < electrolyzer_mw else electrolyzer_mw
            if 0.0 < load < min_load_mw:
                deficit = min_load_mw - load
                available = soc * eta_discharge
                if battery_power_mw < available:
                    available = battery_power_mw
                discharge = deficit if deficit < available else available
                load += discharge
                if load < min_load_mw:
                    # No se alcanza el minimo: se apaga y se revierte la descarga.
                    load = 0.0
                    discharge = 0.0
        else:
            # THRESHOLD: operar solo si el minimo es alcanzable.
            if generation >= min_load_mw and electrolyzer_mw > 0.0:
                load = generation if generation < electrolyzer_mw else electrolyzer_mw
            elif electrolyzer_mw > 0.0 and battery_power_mw > 0.0 and soc > 0.0:
                available = soc * eta_discharge
                if battery_power_mw < available:
                    available = battery_power_mw
                if generation + available >= min_load_mw:
                    headroom = electrolyzer_mw - generation
                    discharge = available if available < headroom else headroom
                    if discharge < 0.0:
                        discharge = 0.0
                    load = generation + discharge
                    if load > electrolyzer_mw:
                        load = electrolyzer_mw

        if discharge > 0.0:
            soc -= discharge / eta_discharge
            if soc < 0.0:
                soc = 0.0

        surplus = generation + discharge - load
        if surplus < 0.0:
            surplus = 0.0

        # Cargar la bateria con el excedente antes de exportar.
        if surplus > 1e-12 and battery_power_mw > 0.0 and battery_energy_mwh > 0.0:
            room = (battery_energy_mwh - soc) / eta_charge
            if room < 0.0:
                room = 0.0
            charge = surplus if surplus < battery_power_mw else battery_power_mw
            if room < charge:
                charge = room
            if charge > 0.0:
                soc += charge * eta_charge
                surplus -= charge
            else:
                charge = 0.0

        # Exportar el remanente, recortando lo que exceda el limite de conexion.
        if surplus > 1e-12:
            if grid_export_limit_mw < 0.0:
                exported = surplus
            else:
                exported = surplus if surplus < grid_export_limit_mw else grid_export_limit_mw
                curtailed = surplus - exported

        electrolyzer_load.append(load)
        grid_sales.append(exported)
        battery_charge.append(charge)
        battery_discharge.append(discharge)
        curtailment.append(curtailed)
        state_of_charge.append(soc)

    return (electrolyzer_load, grid_sales, battery_charge,
            battery_discharge, curtailment, state_of_charge)


def dispatch_hourly(generation_mw, electrolyzer_mw, min_load_ratio, battery_power_mw,
                    battery_energy_mwh, eta_charge, eta_discharge,
                    policy=MinLoadPolicy.TOP_UP, grid_export_limit_mw=None):
    """Despacha un ano horario y devuelve las series resultantes."""
    limit = _NO_GRID_LIMIT if grid_export_limit_mw is None else float(grid_export_limit_mw)
    generation = np.asarray(generation_mw, dtype=np.float64).tolist()

    outputs = _dispatch_core(
        generation,
        float(electrolyzer_mw),
        float(min_load_ratio * electrolyzer_mw),
        float(battery_power_mw),
        float(battery_energy_mwh),
        float(eta_charge),
        float(eta_discharge),
        int(policy),
        limit,
    )
    return DispatchResult(*(np.asarray(s, dtype=np.float64) for s in outputs))

## 9. Simulacion anual

Dos puntos de entrada sobre un unico nucleo, uno por instancia del problema:

**`simulate_base`** -- replica del paper. Electrolizador continuo, bateria dimensionada
como 30% de su capacidad redondeado al pack comercial de 2.5 MW, duracion 1 h, politica
`TOP_UP`.

**`simulate_discrete`** -- extension del trabajo. Electrolizador modular en unidades
enteras de 5 MW, potencia y duracion de bateria como variables de decision, politica
`THRESHOLD`.

Ambos devuelven el mismo `SimulationResult`, de modo que las dos instancias se pueden
comparar en una misma tabla.

**Restricciones.** Antes de simular se chequean cinco condiciones; si alguna falla, no se
simula y el resultado registra el **motivo** del rechazo con LCOE y LCOH en infinito. Que
el motivo quede escrito -- en vez de un `feasible=False` mudo -- es lo que permite
distinguir un diseno que viola la capacidad total de uno que viola el rango discreto.

**AGSR** (*Annual Grid Sales Ratio*) es la restriccion central del paper: la energia
exportada a la red no puede superar el 20% de la generacion renovable total. Es lo que
obliga a que el sistema consuma su propia energia produciendo hidrogeno en vez de
venderla.

In [ ]:
@dataclass(frozen=True)
class SimulationResult:
    """Metricas anuales de una configuracion de capacidades."""

    # -- capacidades --------------------------------------------------------
    wind_mw: float
    pv_mw: float
    electrolyzer_mw: float
    n_electrolyzer_units: int
    battery_mw: float
    battery_mwh: float
    battery_duration_h: float

    # -- factibilidad -------------------------------------------------------
    feasible: bool
    agsr: float
    infeasibility_reason: str = ""

    # -- energia (MWh/ano) --------------------------------------------------
    total_wind_mwh: float = 0.0
    total_pv_mwh: float = 0.0
    total_renewable_mwh: float = 0.0
    total_grid_sales_mwh: float = 0.0
    total_electrolyzer_load_mwh: float = 0.0
    total_curtailment_mwh: float = 0.0
    battery_throughput_mwh: float = 0.0

    # -- desempeno ----------------------------------------------------------
    electrolyzer_cf: float = 0.0
    total_h2_kg: float = 0.0

    # -- economia (CNY) -----------------------------------------------------
    npc_cny: float = 0.0
    annualized_cost_cny: float = 0.0
    lcoe_cny_per_kwh: float = float("inf")
    lcoh_cny_per_kg: float = float("inf")

    def to_dict(self):
        return {f.name: getattr(self, f.name) for f in fields(self)}


def infeasible_result(wind_mw, pv_mw, electrolyzer_mw, n_electrolyzer_units,
                      battery_mw, battery_mwh, battery_duration_h, reason):
    """Resultado nulo para una configuracion que viola una restriccion."""
    return SimulationResult(
        wind_mw=wind_mw, pv_mw=pv_mw, electrolyzer_mw=electrolyzer_mw,
        n_electrolyzer_units=n_electrolyzer_units, battery_mw=battery_mw,
        battery_mwh=battery_mwh, battery_duration_h=battery_duration_h,
        feasible=False, agsr=float("nan"), infeasibility_reason=reason,
    )


_CAPACITY_TOLERANCE_MW = 1e-9


def battery_power_for(electrolyzer_mw, config):
    """Potencia de bateria del modelo base, en tamanos de pack comerciales."""
    battery = config.battery
    exact = battery.power_ratio_to_electrolyzer * electrolyzer_mw
    step = battery.power_rounding_step_mw
    # El epsilon evita que un exacto ya alineado a la malla salte un pack por
    # error de coma flotante (25.0 / 2.5 = 9.999...).
    return math.ceil(exact / step - 1e-9) * step


def _check_base_constraints(wind_mw, pv_mw, electrolyzer_mw, config):
    constraints = config.constraints
    total = wind_mw + pv_mw

    if wind_mw < 0 or pv_mw < 0:
        return "capacidad negativa (wind={:.1f}, pv={:.1f})".format(wind_mw, pv_mw)
    if abs(total - constraints.total_generation_capacity_mw) > _CAPACITY_TOLERANCE_MW:
        return "wind+pv={:.3f} MW no respeta la capacidad fija de {:.1f} MW".format(
            total, constraints.total_generation_capacity_mw)
    if electrolyzer_mw <= 0:
        return "electrolizador de capacidad nula"
    if electrolyzer_mw > constraints.electrolyzer_max_mw + _CAPACITY_TOLERANCE_MW:
        return "electrolizador {:.1f} MW supera el maximo de {:.1f} MW".format(
            electrolyzer_mw, constraints.electrolyzer_max_mw)
    return ""


def _check_discrete_constraints(wind_mw, pv_mw, n_units, battery_mw,
                                battery_duration_h, config):
    electrolyzer = config.electrolyzer
    battery = config.battery

    reason = _check_base_constraints(wind_mw, pv_mw, n_units * electrolyzer.unit_mw, config)
    if reason:
        return reason

    if not electrolyzer.min_units <= n_units <= electrolyzer.max_units:
        return "n_units={} fuera del rango discreto [{}, {}]".format(
            n_units, electrolyzer.min_units, electrolyzer.max_units)
    if not battery.power_min_mw <= battery_mw <= battery.power_max_mw:
        return "bateria {:.1f} MW fuera del rango [{:.1f}, {:.1f}]".format(
            battery_mw, battery.power_min_mw, battery.power_max_mw)
    if not any(abs(battery_duration_h - c) < 1e-9 for c in battery.duration_candidates_h):
        return "duracion de bateria {} h no esta entre las candidatas {}".format(
            battery_duration_h, list(battery.duration_candidates_h))
    return ""


def _simulate(hourly, wind_mw, pv_mw, electrolyzer_mw, n_electrolyzer_units,
              battery_mw, battery_duration_h, config, policy):
    """Nucleo comun: despacha el ano y calcula energia, desempeno y economia."""
    cache = as_profile_cache(hourly, config)
    battery = config.battery
    converter = config.converter
    battery_mwh = battery_mw * battery_duration_h

    wind_profile = cache.wind(wind_mw)
    pv_profile = cache.pv(pv_mw)
    generation = cache.total(wind_mw, pv_mw)

    # El consumo parasito de la planta se descuenta de la generacion bruta antes
    # de despachar. Con el default de 0.0 el perfil pasa intacto.
    if converter.plant_load_ratio > 0:
        generation = generation * (1.0 - converter.plant_load_ratio)

    dispatch = dispatch_hourly(
        generation_mw=generation,
        electrolyzer_mw=electrolyzer_mw,
        min_load_ratio=config.electrolyzer.min_load_ratio,
        battery_power_mw=battery_mw,
        battery_energy_mwh=battery_mwh,
        # La bateria pasa por el convertidor bidireccional en ambos sentidos,
        # ademas de su propia eficiencia de ciclo.
        eta_charge=battery.eta_charge * converter.bidirectional_efficiency,
        eta_discharge=battery.eta_discharge * converter.bidirectional_efficiency,
        policy=policy,
        grid_export_limit_mw=config.constraints.grid_export_limit_mw,
    )

    energy_wind = float(wind_profile.sum())
    energy_pv = float(pv_profile.sum())
    energy_renewable = energy_wind + energy_pv
    energy_grid = float(dispatch.grid_sales.sum())
    energy_electrolyzer = float(dispatch.electrolyzer_load.sum())
    energy_curtailed = float(dispatch.curtailment.sum())
    throughput = float(dispatch.battery_charge.sum() + dispatch.battery_discharge.sum())

    agsr = energy_grid / energy_renewable if energy_renewable > 0 else 0.0
    feasible = agsr <= config.constraints.agsr_max
    reason = "" if feasible else "AGSR={:.4f} supera el maximo de {:.2f}".format(
        agsr, config.constraints.agsr_max)

    hours = cache.n_hours
    capacity_factor = (energy_electrolyzer / (electrolyzer_mw * hours)
                       if electrolyzer_mw > 0 else 0.0)

    # El electrolizador se alimenta en DC: la energia que llega a las celdas es la
    # que toma del bus AC menos la perdida del rectificador. El factor de capacidad
    # sigue midiendose sobre la carga AC, que es la potencia contratada.
    energy_to_cells = energy_electrolyzer * converter.rectifier_efficiency
    h2_kg = energy_to_cells * 1000.0 / config.electrolyzer.specific_consumption_kwh_per_kg

    npc = npc_from_capacities(
        wind_mw=wind_mw, pv_mw=pv_mw, electrolyzer_mw=electrolyzer_mw,
        battery_mw=battery_mw, battery_mwh=battery_mwh,
        costs=config.costs, economics=config.economics,
        hydrogen_storage_kg=config.electrolyzer.hydrogen_storage_kg,
    )
    annualized = npc * crf(config.economics.real_discount_rate,
                           config.economics.project_lifetime_years)

    if config.economics.lcoe_energy_basis == "electrolyzer_only":
        served_mwh = energy_electrolyzer
    else:
        served_mwh = energy_electrolyzer + energy_grid

    return SimulationResult(
        wind_mw=wind_mw, pv_mw=pv_mw, electrolyzer_mw=electrolyzer_mw,
        n_electrolyzer_units=n_electrolyzer_units, battery_mw=battery_mw,
        battery_mwh=battery_mwh, battery_duration_h=battery_duration_h,
        feasible=bool(feasible), agsr=float(agsr), infeasibility_reason=reason,
        total_wind_mwh=energy_wind, total_pv_mwh=energy_pv,
        total_renewable_mwh=energy_renewable, total_grid_sales_mwh=energy_grid,
        total_electrolyzer_load_mwh=energy_electrolyzer,
        total_curtailment_mwh=energy_curtailed, battery_throughput_mwh=throughput,
        electrolyzer_cf=float(capacity_factor), total_h2_kg=float(h2_kg),
        npc_cny=float(npc), annualized_cost_cny=float(annualized),
        lcoe_cny_per_kwh=levelized_cost(annualized, served_mwh * 1000.0),
        lcoh_cny_per_kg=levelized_cost(annualized, h2_kg),
    )


def simulate_base(hourly, wind_mw, pv_mw, electrolyzer_mw, config,
                  battery_duration_h=None):
    """Modelo base del paper, con bateria proporcional al electrolizador."""
    battery = config.battery
    duration_h = battery.duration_h if battery_duration_h is None else battery_duration_h
    battery_mw = battery_power_for(electrolyzer_mw, config)
    battery_mwh = battery_mw * duration_h
    units = int(round(electrolyzer_mw / config.electrolyzer.unit_mw))

    reason = _check_base_constraints(wind_mw, pv_mw, electrolyzer_mw, config)
    if reason:
        return infeasible_result(wind_mw, pv_mw, electrolyzer_mw, units,
                                 battery_mw, battery_mwh, duration_h, reason)

    return _simulate(hourly, wind_mw, pv_mw, electrolyzer_mw, units, battery_mw,
                     duration_h, config, MinLoadPolicy.TOP_UP)


def simulate_discrete(hourly, wind_mw, n_electrolyzer_units, battery_mw,
                      battery_duration_h, config):
    """Extension discreta: electrolizador modular y bateria libre."""
    total_capacity = config.constraints.total_generation_capacity_mw
    pv_mw = total_capacity - wind_mw
    electrolyzer_mw = n_electrolyzer_units * config.electrolyzer.unit_mw
    battery_mwh = battery_mw * battery_duration_h

    reason = _check_discrete_constraints(wind_mw, pv_mw, n_electrolyzer_units,
                                         battery_mw, battery_duration_h, config)
    if reason:
        return infeasible_result(wind_mw, pv_mw, electrolyzer_mw,
                                 int(n_electrolyzer_units), battery_mw,
                                 battery_mwh, battery_duration_h, reason)

    return _simulate(hourly, wind_mw, pv_mw, electrolyzer_mw, int(n_electrolyzer_units),
                     battery_mw, battery_duration_h, config, MinLoadPolicy.THRESHOLD)

In [ ]:
# Comprobacion del simulador sobre el caso optimo del paper: W190-P10-E95-B30.
caso = simulate_base(cache_demo, 190.0, 10.0, 95.0, ESCENARIO)

display(pd.DataFrame([{
    "caso": "W190-P10-E95-B30",
    "bateria_mw": caso.battery_mw,
    "AGSR": caso.agsr,
    "factible": caso.feasible,
    "CF electrolizador": caso.electrolyzer_cf,
    "H2 [t/ano]": caso.total_h2_kg / 1000.0,
    "NPC [M CNY]": caso.npc_cny / 1e6,
    "LCOE [CNY/kWh]": caso.lcoe_cny_per_kwh,
    "LCOH [CNY/kg]": caso.lcoh_cny_per_kg,
}]).T.rename(columns={0: "valor"}))

# Un diseno que viola una restriccion no se simula: queda el motivo escrito.
rechazado = simulate_base(cache_demo, 190.0, 10.0, 120.0, ESCENARIO)
print("Diseno rechazado ->", rechazado.infeasibility_reason)

## 10. Validacion contra las cifras publicadas

**Aqui se juega la credibilidad de todo lo que sigue.** Antes de afirmar que una
metaheuristica mejora al paper hay que demostrar que el modelo reproduce al paper; si no,
no hay forma de saber si la "mejora" viene del algoritmo o de estar resolviendo un
problema distinto.

Se contrastan **10 objetivos** extraidos del PDF, cada uno con su seccion de origen y su
tolerancia. Las tolerancias son mas anchas donde el paper no publica el detalle del modelo
(el recurso: 8% y 15%) y mas estrictas en lo economico (3%), que depende solo de la
Tabla 4 y del horizonte.

Un `REVISAR` no es necesariamente un error del modelo. El caso conocido es
**NPC W60-P140-E85-B27.5**: los propios costos de la Tabla 4 del paper implican un delta
de -26.7 M CNY respecto de W120-P80, no los -235.9 que reporta el texto. Los otros dos NPC
calzan al 0.4%, de modo que la inconsistencia esta en el paper, no en la replicacion.

In [ ]:
@dataclass(frozen=True)
class Target:
    """Un valor publicado contra el que se contrasta la replicacion."""

    metric: str
    reported: float
    section: str
    unit: str = ""
    tolerance_pct: float = 3.0
    note: str = ""

    def row(self, replicated):
        if replicated is None or not np.isfinite(replicated):
            deviation = np.nan
            verdict = "sin dato"
        else:
            deviation = (replicated - self.reported) / self.reported * 100.0
            verdict = "OK" if abs(deviation) <= self.tolerance_pct else "REVISAR"
        return {
            "metrica": self.metric, "unidad": self.unit, "paper": self.reported,
            "replicacion": replicated, "desvio_pct": deviation, "veredicto": verdict,
            "seccion": self.section, "nota": self.note,
        }


PAPER_TARGETS = {
    "wind_cf": Target("CF parque eolico", 0.40, "3.1", "fraccion", 8.0),
    "pv_cf": Target("CF parque fotovoltaico", 0.19, "3.1", "fraccion", 15.0),
    "electrolyzer_cf": Target("CF electrolizador (W190-P10-E95)", 0.6240, "3.1",
                              "fraccion", 8.0),
    "lcoe_w190": Target("LCOE W190-P10-E95-B30", 0.2692, "3.2", "CNY/kWh", 3.0),
    "lcoe_w120": Target("LCOE W120-P80-E80-B25", 0.2886, "3.2", "CNY/kWh", 3.0),
    "lcoe_w60": Target("LCOE W60-P140-E85-B27.5", 0.3617, "3.2", "CNY/kWh", 3.0),
    "npc_w190": Target("NPC W190-P10-E95-B30", 2675.73, "3.2", "M CNY", 3.0),
    "npc_w120": Target("NPC W120-P80-E80-B25", 2350.38, "3.2", "M CNY", 3.0),
    "npc_w60": Target(
        "NPC W60-P140-E85-B27.5", 2114.53, "3.2", "M CNY", 3.0,
        note=("inconsistente con la Tabla 4 del paper: sus propios costos implican un "
              "delta de -26.7 M CNY respecto de W120-P80, no los -235.9 que reporta."),
    ),
    "capex_share": Target("CAPEX sobre NPC (W190)", 0.7330, "3.2", "fraccion", 5.0),
}

# Los tres casos con nombre del paper: (etiqueta, wind_mw, pv_mw, electrolyzer_mw).
NAMED_CASES = (
    ("w190", 190.0, 10.0, 95.0),
    ("w120", 120.0, 80.0, 80.0),
    ("w60", 60.0, 140.0, 85.0),
)


def resource_metrics(hourly, config):
    """Factores de capacidad del recurso, independientes del dimensionamiento.

    Se calculan sobre un parque de referencia y se normalizan por su capacidad, de modo
    que el CF no dependa de cuantos MW se instalen.
    """
    cache = as_profile_cache(hourly, config)
    hours = cache.n_hours
    reference_mw = 100.0

    return {
        "wind_cf": float(cache.wind(reference_mw).sum()) / (reference_mw * hours),
        "pv_cf": float(cache.pv(reference_mw).sum()) / (reference_mw * hours),
        "mean_wind_speed_50m": float(np.mean(hourly["ws50m"])),
        "mean_hub_wind_speed": float(np.mean(hourly["wsc_ms"]))
        if "wsc_ms" in hourly.columns else np.nan,
        "annual_ghi_kwh_m2": float(np.sum(hourly["ghi_kwh_m2"])),
        "wind_column": cache.wind_column,
    }


def _capex_share(result, config):
    """Fraccion del NPC que corresponde a inversion inicial.

    El paper reporta 73.30% para el caso optimo, lo que valida indirectamente el reparto
    entre capex, reemplazos, O&M y residual.
    """
    costs = config.costs
    capex = (
        result.wind_mw * 1000.0 * costs.wind.capex_cny_per_kw
        + result.pv_mw * 1000.0 * costs.pv.capex_cny_per_kw
        + result.electrolyzer_mw * 1000.0 * costs.electrolyzer.capex_cny_per_kw
        + result.battery_mw * 1000.0 * costs.battery.capex_cny_per_kw
        + config.electrolyzer.hydrogen_storage_kg * costs.hydrogen_storage_capex_cny_per_kg
    )
    return capex / result.npc_cny if result.npc_cny > 0 else float("nan")


def validate_scenario(hourly, config):
    """Contrasta la replicacion completa contra los objetivos publicados."""
    cache = as_profile_cache(hourly, config)
    frame = hourly if isinstance(hourly, pd.DataFrame) else None

    results = {label: simulate_base(cache, wind, pv, electrolyzer, config)
               for label, wind, pv, electrolyzer in NAMED_CASES}

    replicated = {
        "electrolyzer_cf": results["w190"].electrolyzer_cf,
        "lcoe_w190": results["w190"].lcoe_cny_per_kwh,
        "lcoe_w120": results["w120"].lcoe_cny_per_kwh,
        "lcoe_w60": results["w60"].lcoe_cny_per_kwh,
        "npc_w190": results["w190"].npc_cny / 1e6,
        "npc_w120": results["w120"].npc_cny / 1e6,
        "npc_w60": results["w60"].npc_cny / 1e6,
        "capex_share": _capex_share(results["w190"], config),
    }

    if frame is not None:
        resource = resource_metrics(frame, config)
        replicated["wind_cf"] = resource["wind_cf"]
        replicated["pv_cf"] = resource["pv_cf"]

    return pd.DataFrame([target.row(replicated.get(key))
                         for key, target in PAPER_TARGETS.items()])


validacion = validate_scenario(hourly, ESCENARIO)
display(validacion.drop(columns=["nota"]))

n_ok = int((validacion["veredicto"] == "OK").sum())
n_con_dato = int((validacion["veredicto"] != "sin dato").sum())
display(Markdown(
    "**{} de {} objetivos publicados quedan dentro de tolerancia.**".format(n_ok, n_con_dato)
))
for _, fila in validacion[validacion["nota"] != ""].iterrows():
    print("Nota sobre '{}': {}".format(fila["metrica"], fila["nota"]))

## 11. Instancia base: barrido exhaustivo

Antes de traer una metaheuristica hay que saber contra que se la compara. En la instancia
base el dominio es lo bastante chico para **enumerarlo completo**, de modo que el optimo
global se conoce con certeza:

- capacidad eolica: 0 a 200 MW en pasos de 10 MW (**21 valores**), con PV = 200 - eolica
- electrolizador: 5 a 100 MW en pasos de 5 MW (**20 valores**)
- bateria: 30% del electrolizador, redondeada al pack de 2.5 MW, 1 h

Son **420 configuraciones**. Este barrido es tambien el motor del estudio de sensibilidad
de la seccion 19: al ser determinista, sus curvas no llevan ruido de semilla.

In [ ]:
def grid_axes(config):
    """Ejes de la malla: capacidades de viento y de electrolizador."""
    total_capacity = config.constraints.total_generation_capacity_mw
    wind_values = np.arange(0.0, total_capacity + 1e-9, config.search.wind_step_mw)
    electrolyzer_values = np.arange(
        config.search.electrolyzer_step_mw,
        config.constraints.electrolyzer_max_mw + 1e-9,
        config.search.electrolyzer_step_mw,
    )
    return wind_values, electrolyzer_values


def run_grid_search(hourly, config):
    """Evalua toda la malla y devuelve una fila por configuracion."""
    cache = as_profile_cache(hourly, config)
    wind_values, electrolyzer_values = grid_axes(config)
    total_capacity = config.constraints.total_generation_capacity_mw

    rows = []
    for wind_mw in wind_values:
        pv_mw = total_capacity - float(wind_mw)
        if pv_mw < 0:
            continue
        for electrolyzer_mw in electrolyzer_values:
            result = simulate_base(cache, float(wind_mw), pv_mw,
                                   float(electrolyzer_mw), config)
            rows.append(result.to_dict())
    return pd.DataFrame(rows)


def summarize_grid(results):
    """Conteos y mejor solucion de un barrido."""
    feasible = results[results["feasible"]]
    summary = {
        "evaluated": int(len(results)),
        "feasible": int(len(feasible)),
        "infeasible": int(len(results) - len(feasible)),
    }
    if len(feasible) > 0:
        best = feasible.sort_values("lcoe_cny_per_kwh").iloc[0]
        summary["best_lcoe_cny_per_kwh"] = float(best["lcoe_cny_per_kwh"])
        summary["best_wind_mw"] = float(best["wind_mw"])
        summary["best_electrolyzer_mw"] = float(best["electrolyzer_mw"])
        summary["best_electrolyzer_cf"] = float(best["electrolyzer_cf"])
    return summary


def best_solution(results, objective="lcoe_cny_per_kwh"):
    """Mejor fila factible segun el objetivo, o None si no hay ninguna."""
    feasible = results[results["feasible"]]
    if len(feasible) == 0:
        return None
    return feasible.sort_values(objective).iloc[0]


t0 = time.perf_counter()
base_grid = run_grid_search(hourly, ESCENARIO)
print("Barrido base: {} configuraciones en {:.1f} s".format(
    len(base_grid), time.perf_counter() - t0))

resumen_base = summarize_grid(base_grid)
base_best = best_solution(base_grid)

print("Factibles   : {} de {} ({:.1%})".format(
    resumen_base["feasible"], resumen_base["evaluated"],
    resumen_base["feasible"] / resumen_base["evaluated"]))

display(base_grid[base_grid["feasible"]]
        .sort_values("lcoe_cny_per_kwh")
        .head(10)[["wind_mw", "pv_mw", "electrolyzer_mw", "battery_mw", "agsr",
                   "electrolyzer_cf", "total_h2_kg", "npc_cny", "lcoe_cny_per_kwh",
                   "lcoh_cny_per_kg"]])

display(Markdown(
    "**Optimo global de la instancia base (por enumeracion):** "
    "W={:.0f} MW, PV={:.0f} MW, E={:.0f} MW, B={:.1f} MW / 1 h -- "
    "LCOE **{:.4f} CNY/kWh**, AGSR {:.4f}, CF electrolizador {:.4f}.".format(
        base_best.wind_mw, base_best.pv_mw, base_best.electrolyzer_mw,
        base_best.battery_mw, base_best.lcoe_cny_per_kwh, base_best.agsr,
        base_best.electrolyzer_cf)
))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel 1: LCOE sobre todo el dominio, distinguiendo factibles de infactibles.
infactibles = base_grid[~base_grid["feasible"]]
factibles = base_grid[base_grid["feasible"]]

axes[0].scatter(infactibles["wind_mw"], infactibles["electrolyzer_mw"],
                c="#d9d5cf", s=42, marker="s", label="infactible (AGSR > 20%)")
sc = axes[0].scatter(factibles["wind_mw"], factibles["electrolyzer_mw"],
                     c=factibles["lcoe_cny_per_kwh"], cmap="viridis", s=48, marker="s")
axes[0].scatter([base_best.wind_mw], [base_best.electrolyzer_mw], marker="*",
                s=320, color="#c62828", edgecolor="white", linewidth=0.8,
                label="optimo global", zorder=5)
axes[0].set_xlabel("Capacidad eolica [MW]   (PV = 200 - eolica)")
axes[0].set_ylabel("Electrolizador [MW]")
axes[0].set_title("Dominio evaluado y LCOE de las soluciones factibles")
axes[0].legend(loc="lower right", framealpha=0.95)
axes[0].grid(False)
fig.colorbar(sc, ax=axes[0], label="LCOE [CNY/kWh]")

# Panel 2: AGSR contra capacidad eolica, que es lo que define la frontera factible.
for e_mw, grupo in base_grid.groupby("electrolyzer_mw"):
    if e_mw % 20 == 0:
        axes[1].plot(grupo["wind_mw"], grupo["agsr"], marker="o", ms=3, lw=1.2,
                     label="E = {:.0f} MW".format(e_mw))
axes[1].axhline(ESCENARIO.constraints.agsr_max, color="#c62828", ls="--", lw=1.6,
                label="limite AGSR = 20%")
axes[1].set_xlabel("Capacidad eolica [MW]")
axes[1].set_ylabel("AGSR [fraccion]")
axes[1].set_title("La restriccion de AGSR define la frontera factible")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 12. Instancia discreta: espacio de decision y funcion objetivo

La extension del trabajo cambia el problema en tres puntos respecto de la instancia base:

1. El electrolizador se compra en **unidades enteras** de 5 MW (entre 10 y 20 unidades,
   es decir 50 a 100 MW), no como una capacidad continua.
2. La **potencia de bateria** es una variable libre en multiplos de 5 MW (0 a 50 MW), no
   un porcentaje fijo del electrolizador.
3. La **duracion de bateria** es una variable de decision del catalogo {1, 2, 4} h.

Esto convierte el problema en uno **mixto**: una variable continua, dos enteras y una
categorica. El vector de decision tiene 4 componentes:

| | variable | tipo | rango |
|---|---|---|---|
| `x[0]` | capacidad eolica [MW] | continua | [0, 200], con PV = 200 - x[0] |
| `x[1]` | unidades de electrolizador | entera | [10, 20] |
| `x[2]` | potencia de bateria [MW] | discreta, paso 5 | [0, 50] |
| `x[3]` | indice del catalogo de duraciones | entera | [0, 2] |

**Todas las metaheuristicas trabajan sobre un vector continuo**, y la proyeccion al
espacio mixto real ocurre en un solo lugar: `DecisionSpace.decode`. Centralizarlo importa
-- mientras los limites vivan dentro de cada algoritmo, agregar uno nuevo obliga a
reescribirlos, y cualquier discrepancia hace que dos algoritmos exploren espacios
distintos y sus resultados dejen de ser comparables.

### La penalizacion

Una solucion infactible (AGSR > 20%) no se descarta: recibe

$$score = 10^6 + 10^3 \cdot \max(AGSR - 0.20,\ 0)$$

La penalizacion es **proporcional a cuanto se viola** la restriccion. Un descarte plano
dejaria al optimizador sin informacion sobre cuan lejos esta de cumplir -- todas las
soluciones malas se verian igual de malas -- mientras que asi hay gradiente hacia la
region factible.

In [ ]:
@dataclass(frozen=True)
class Design:
    """Un diseno concreto, ya proyectado al espacio factible."""

    wind_mw: float
    pv_mw: float
    n_electrolyzer_units: int
    electrolyzer_mw: float
    battery_mw: float
    battery_duration_h: float
    battery_mwh: float

    def to_dict(self):
        return {f.name: getattr(self, f.name) for f in fields(self)}


class DecisionSpace:
    """Limites y decodificacion del vector de decision."""

    VARIABLE_NAMES = ("wind_mw", "n_electrolyzer_units", "battery_mw",
                      "battery_duration_index")

    def __init__(self, config):
        self.config = config
        electrolyzer = config.electrolyzer
        battery = config.battery
        constraints = config.constraints

        self.lower = np.array(
            [0.0, electrolyzer.min_units, battery.power_min_mw, 0.0], dtype=float)
        self.upper = np.array(
            [constraints.total_generation_capacity_mw,
             electrolyzer.max_units,
             battery.power_max_mw,
             len(battery.duration_candidates_h) - 1], dtype=float)

    @property
    def n_dimensions(self):
        return len(self.lower)

    def clip(self, x):
        """Recorta un vector para que permanezca dentro de los limites."""
        return np.clip(np.asarray(x, dtype=float), self.lower, self.upper)

    def sample(self, rng, size):
        """Poblacion inicial uniforme dentro de los limites."""
        return rng.uniform(self.lower, self.upper, size=(size, self.n_dimensions))

    def decode(self, x):
        """Proyecta un vector continuo al diseno discreto mas cercano."""
        config = self.config
        battery = config.battery
        electrolyzer = config.electrolyzer
        total_capacity = config.constraints.total_generation_capacity_mw

        x = self.clip(x)

        wind_mw = float(x[0])
        pv_mw = total_capacity - wind_mw

        units = int(np.clip(round(float(x[1])),
                            electrolyzer.min_units, electrolyzer.max_units))

        step = battery.power_step_mw
        battery_mw = float(np.clip(round(float(x[2]) / step) * step,
                                   battery.power_min_mw, battery.power_max_mw))

        candidates = battery.duration_candidates_h
        index = int(np.clip(round(float(x[3])), 0, len(candidates) - 1))
        duration_h = float(candidates[index])

        return Design(
            wind_mw=wind_mw, pv_mw=pv_mw, n_electrolyzer_units=units,
            electrolyzer_mw=units * electrolyzer.unit_mw, battery_mw=battery_mw,
            battery_duration_h=duration_h, battery_mwh=battery_mw * duration_h,
        )

    def warnings(self):
        """Avisos de modelado, no errores de configuracion.

        El caso importante: si la duracion de bateria es variable de decision pero no
        tiene costo por energia, alargarla es gratis y el optimizador la llevara al
        maximo sin que ese resultado signifique nada. Por eso el escenario reparte el
        costo de la bateria 30/70 entre potencia y energia (seccion 4).
        """
        messages = []
        battery = self.config.battery
        if (len(battery.duration_candidates_h) > 1
                and not self.config.costs.battery.has_energy_cost):
            messages.append(
                "la duracion de bateria es variable de decision ({} candidatas) pero "
                "costs.battery no tiene componente por kWh: alargar la bateria no cuesta "
                "nada y el optimo en duracion no es interpretable.".format(
                    len(battery.duration_candidates_h))
            )
        return messages


@dataclass(frozen=True)
class PenaltyPolicy:
    """Como se castiga la infactibilidad."""

    base: float = 1.0e6
    agsr_weight: float = 1.0e3

    @classmethod
    def from_config(cls, config):
        return cls(base=config.penalty_infeasible,
                   agsr_weight=config.penalty_agsr_weight)

    def score(self, result, agsr_max):
        """Puntaje a minimizar: LCOE si es factible, penalizacion si no."""
        if result.feasible:
            return result.lcoe_cny_per_kwh

        # Un AGSR NaN indica que la solucion ni siquiera llego a simularse (violo una
        # restriccion de capacidad); recibe solo la penalizacion base.
        if np.isnan(result.agsr):
            return self.base

        excess = max(result.agsr - agsr_max, 0.0)
        return self.base + self.agsr_weight * excess


@dataclass(frozen=True)
class Evaluation:
    """Una evaluacion completa: puntaje, diseno decodificado y resultado."""

    score: float
    design: Design
    result: SimulationResult

    def to_record(self):
        record = self.result.to_dict()
        record["score"] = self.score
        return record


class ObjectiveFunction:
    """Objetivo evaluable, con cache de perfiles y conteo de evaluaciones.

    Todos los optimizadores comparten esta instancia, de modo que compiten con el mismo
    modelo, el mismo cache y el mismo contador de presupuesto.
    """

    def __init__(self, hourly, config, space=None, penalty=None):
        self.config = config
        self.cache = as_profile_cache(hourly, config)
        self.space = space if space is not None else DecisionSpace(config)
        self.penalty = (penalty if penalty is not None
                        else PenaltyPolicy.from_config(config.metaheuristic))
        self.n_evaluations = 0

    def __call__(self, x):
        return self.evaluate(x)

    def evaluate(self, x):
        design = self.space.decode(x)
        result = simulate_discrete(
            self.cache,
            wind_mw=design.wind_mw,
            n_electrolyzer_units=design.n_electrolyzer_units,
            battery_mw=design.battery_mw,
            battery_duration_h=design.battery_duration_h,
            config=self.config,
        )
        self.n_evaluations += 1
        score = self.penalty.score(result, self.config.constraints.agsr_max)
        return Evaluation(score=float(score), design=design, result=result)

    def reset_counter(self):
        self.n_evaluations = 0


ESPACIO = DecisionSpace(ESCENARIO)

for aviso in ESPACIO.warnings():
    print("AVISO:", aviso)

display(pd.DataFrame({
    "variable": ESPACIO.VARIABLE_NAMES,
    "limite_inferior": ESPACIO.lower,
    "limite_superior": ESPACIO.upper,
    "proyeccion": ["continua (PV = 200 - x0)",
                   "entero mas cercano",
                   "multiplo de {:.0f} MW".format(ESCENARIO.battery.power_step_mw),
                   "indice del catalogo {}".format(
                       list(ESCENARIO.battery.duration_candidates_h))],
}))

# Tamano del espacio discreto efectivo, para dimensionar el presupuesto.
n_wind = 41   # discretizacion de referencia de la variable continua, pasos de 5 MW
n_units = ESCENARIO.electrolyzer.max_units - ESCENARIO.electrolyzer.min_units + 1
n_batt = int(ESCENARIO.battery.power_max_mw / ESCENARIO.battery.power_step_mw) + 1
n_dur = len(ESCENARIO.battery.duration_candidates_h)
print("Combinaciones discretas (con la eolica en pasos de 5 MW): {:,}".format(
    n_wind * n_units * n_batt * n_dur))
print("Presupuesto de una corrida: {} evaluaciones".format(
    ESCENARIO.metaheuristic.evaluation_budget))

## 13. Clase base de los optimizadores

Todo lo que **no** es el operador de busqueda vive aqui: la semilla, el conteo de
evaluaciones, el registro del historial por iteracion y el cronometraje. Agregar una
metaheuristica nueva significa escribir una subclase que implemente `_search` y
registrarla en `REGISTRY`; nada mas.

Que esto sea comun no es una comodidad de programacion, es lo que hace valida la
comparacion: los 6 algoritmos producen historiales con el mismo formato, cuentan las
evaluaciones de la misma forma y se cronometran con el mismo reloj.

**El contrato de una subclase** son dos llamadas:

- `self._evaluate(x)` evalua un vector y actualiza la mejor solucion conocida.
- `self._record(iteration, **extra)` anota el estado de la mejor solucion al cierre de
  una iteracion. Los `extra` sirven para guardar el coeficiente de exploracion del
  algoritmo (`a` en GWO y WOA, `h` en SHO), que es lo que permite despues graficar la
  transicion de exploracion a explotacion.

In [ ]:
@dataclass
class OptimizationResult:
    """Resultado de una corrida: la mejor solucion y como se llego a ella."""

    algorithm: str
    seed: int
    best: Evaluation
    history: pd.DataFrame
    n_evaluations: int
    elapsed_s: float

    @property
    def best_score(self):
        return self.best.score

    def summary(self):
        """Fila plana con lo esencial, para agregar varias corridas."""
        record = self.best.to_record()
        record.update({"algorithm": self.algorithm, "seed": self.seed,
                       "n_evaluations": self.n_evaluations,
                       "elapsed_s": self.elapsed_s})
        return record


class Optimizer:
    """Base de todos los optimizadores sobre el espacio discreto WPEB."""

    name = "base"

    def __init__(self, objective, config=None, seed=None):
        self.objective = objective
        self.config = config if config is not None else objective.config.metaheuristic
        self.seed = int(self.config.seed if seed is None else seed)
        self.rng = np.random.default_rng(self.seed)
        self.space = objective.space
        self.best = None
        self._history = []

    # -- utilidades para las subclases --------------------------------------

    def _evaluate(self, x):
        """Evalua un vector y actualiza la mejor solucion conocida."""
        evaluation = self.objective.evaluate(x)
        if self.best is None or evaluation.score < self.best.score:
            self.best = evaluation
        return evaluation

    def _record(self, iteration, **extra):
        """Anota el estado de la mejor solucion al cierre de una iteracion."""
        if self.best is None:
            raise RuntimeError("no se puede registrar antes de la primera evaluacion")

        result = self.best.result
        row = {
            "iteration": iteration,
            "best_score": self.best.score,
            "n_evaluations": self.objective.n_evaluations,
            "wind_mw": result.wind_mw,
            "pv_mw": result.pv_mw,
            "electrolyzer_mw": result.electrolyzer_mw,
            "n_electrolyzer_units": result.n_electrolyzer_units,
            "battery_mw": result.battery_mw,
            "battery_mwh": result.battery_mwh,
            "battery_duration_h": result.battery_duration_h,
            "feasible": result.feasible,
            "agsr": result.agsr,
            "electrolyzer_cf": result.electrolyzer_cf,
            "lcoe_cny_per_kwh": result.lcoe_cny_per_kwh,
            "lcoh_cny_per_kg": result.lcoh_cny_per_kg,
        }
        row.update(extra)
        self._history.append(row)

    # -- contrato -----------------------------------------------------------

    def _search(self):
        """Ejecuta la busqueda, llamando a ``_evaluate`` y ``_record``."""
        raise NotImplementedError("cada metaheuristica debe implementar _search()")

    def optimize(self):
        """Corre el algoritmo y devuelve el resultado con su historial."""
        self.objective.reset_counter()
        self.best = None
        self._history = []

        started = time.perf_counter()
        self._search()
        elapsed = time.perf_counter() - started

        if self.best is None:
            raise RuntimeError("{} termino sin evaluar ninguna solucion".format(self.name))

        return OptimizationResult(
            algorithm=self.name, seed=self.seed, best=self.best,
            history=pd.DataFrame(self._history),
            n_evaluations=self.objective.n_evaluations, elapsed_s=elapsed,
        )

## 14. Las metaheuristicas

Las seis estan programadas completas a continuacion, **una por celda**, cada una con su
explicacion. Todas leen sus hiperparametros del bloque de configuracion de la seccion 3.

| Algoritmo | Familia | Idea central | Fuente |
|---|---|---|---|
| **GWO** | enjambre jerarquico | jerarquia alfa-beta-delta: cada lobo se mueve al promedio de los tirones de los tres lideres | Mirjalili et al. (2014) |
| **PSO** | enjambre | cada particula combina su mejor historico con el mejor global, con constriccion | Clerc & Kennedy (2002) |
| **GA** | evolutivo | seleccion por torneo, cruce BLX-alpha, mutacion gaussiana, elitismo | real-coded GA estandar |
| **SHO** | enjambre jerarquico | encierro de toda la manada sobre un unico lider (la presa) | Dhiman & Kumar (2017) |
| **WOA** | enjambre | encierro o espiral "bubble-net", sorteados 50/50 por agente | Mirjalili & Lewis (2016) |
| **random** | linea base | muestreo uniforme sobre la malla discreta, sin memoria | -- |

**Sobre la linea base.** `random` no es relleno: es el control del experimento. Toda
metaheuristica deberia superarla con el mismo presupuesto de evaluaciones. Si no lo hace,
el problema esta mal planteado o el algoritmo mal ajustado, y esa es una conclusion tan
valida como cualquier otra.

### 14.1 GWO -- Grey Wolf Optimizer

Modela la jerarquia social de una manada de lobos: las tres mejores posiciones conocidas
son **alfa**, **beta** y **delta**, y el resto de la manada se reposiciona siguiendo a
las tres a la vez.

Para cada lider $\ell$ y cada lobo $i$:

$$\vec{D}_\ell = |\vec{C} \cdot \vec{X}_\ell - \vec{X}_i| \qquad
\vec{X}_{i,\ell} = \vec{X}_\ell - \vec{A} \cdot \vec{D}_\ell$$

y la nueva posicion es el **promedio de los tres tirones**. Los coeficientes son
$\vec{A} = 2a\vec{r_1} - a$ y $\vec{C} = 2\vec{r_2}$, con $\vec{r_1}, \vec{r_2}$
uniformes en $[0,1]$.

El coeficiente $a$ **decae linealmente de 2 a 0** a lo largo de las iteraciones. Cuando
$a$ es grande, $|A|$ puede superar 1 y el lobo se aleja del lider: exploracion. Cuando
$a$ se acerca a 0, $|A| < 1$ siempre y la manada converge sobre los lideres:
explotacion. Esa transicion programada es todo el mecanismo de balance del algoritmo.

El movimiento esta vectorizado sobre lobos y dimensiones: `leaders[:, None, :]` difunde
los 3 lideres sobre toda la manada.

In [ ]:
class GreyWolfOptimizer(Optimizer):
    """GWO clasico: la jerarquia alfa-beta-delta guia al resto de la manada."""

    name = "gwo"

    def _search(self):
        space = self.space
        n_wolves = self.config.population
        n_iterations = self.config.iterations

        positions = space.sample(self.rng, n_wolves)

        # Las tres mejores posiciones conocidas. Se guardan aparte de ``self.best``
        # porque el movimiento del GWO necesita las tres.
        leaders = np.zeros((3, space.n_dimensions), dtype=float)
        leader_scores = np.full(3, np.inf)

        for iteration in range(n_iterations):
            for i in range(n_wolves):
                positions[i] = space.clip(positions[i])
                evaluation = self._evaluate(positions[i])
                self._update_leaders(leaders, leader_scores, positions[i],
                                     evaluation.score)

            # ``a`` decae linealmente de 2 a 0: la manada pasa de explorar a
            # explotar alrededor de los lideres.
            a = 2.0 - iteration * (2.0 / max(n_iterations - 1, 1))

            positions = self._move(positions, leaders, a)

            self._record(iteration + 1, a=a)

    @staticmethod
    def _update_leaders(leaders, leader_scores, position, score):
        """Inserta una posicion en la jerarquia alfa-beta-delta si corresponde."""
        if score < leader_scores[0]:
            leaders[2] = leaders[1]
            leader_scores[2] = leader_scores[1]
            leaders[1] = leaders[0]
            leader_scores[1] = leader_scores[0]
            leaders[0] = position.copy()
            leader_scores[0] = score
        elif score < leader_scores[1]:
            leaders[2] = leaders[1]
            leader_scores[2] = leader_scores[1]
            leaders[1] = position.copy()
            leader_scores[1] = score
        elif score < leader_scores[2]:
            leaders[2] = position.copy()
            leader_scores[2] = score

    def _move(self, positions, leaders, a):
        """Nueva posicion de cada lobo: promedio de los tres tirones de lider."""
        n_wolves, n_dimensions = positions.shape
        shape = (3, n_wolves, n_dimensions)

        r1 = self.rng.random(shape)
        r2 = self.rng.random(shape)

        A = 2.0 * a * r1 - a
        C = 2.0 * r2

        # leaders[:, None, :] difunde los 3 lideres sobre toda la manada.
        distance = np.abs(C * leaders[:, None, :] - positions[None, :, :])
        candidates = leaders[:, None, :] - A * distance

        return candidates.mean(axis=0)

### 14.2 PSO -- Particle Swarm Optimization

Cada particula recuerda su mejor posicion historica (`pbest`) y conoce la mejor de todo
el enjambre (`gbest`). La velocidad combina ambas atracciones:

$$\vec{v} \leftarrow \chi\,\Big(\vec{v}
  + c_1 \vec{r_1} \cdot (\vec{p}_{best} - \vec{x})
  + c_2 \vec{r_2} \cdot (\vec{g}_{best} - \vec{x})\Big)
\qquad \vec{x} \leftarrow \vec{x} + \vec{v}$$

Esta es la formulacion de **constriccion** de Clerc-Kennedy, no la de peso de inercia
clasico: el factor $\chi = 0.729$ multiplica **toda la suma**, no solo la velocidad
previa. Esa diferencia es lo que garantiza convergencia sin necesidad de decaer $\chi$ a
lo largo de las iteraciones, y por eso PSO es el unico de los cinco que no tiene un
coeficiente que decae con el tiempo.

La velocidad se acota al 20% del rango de cada variable (`velocity_clamp_ratio`) para que
una particula no salte de un extremo al otro del dominio en un solo paso.

In [ ]:
class ParticleSwarm(Optimizer):
    """PSO con constriccion de Clerc-Kennedy y velocidad acotada por variable."""

    name = "pso"

    def _search(self):
        space = self.space
        params = self.config.pso
        n_particles = self.config.population
        n_iterations = self.config.iterations

        positions = space.sample(self.rng, n_particles)
        velocity_range = params.velocity_clamp_ratio * (space.upper - space.lower)
        velocities = self.rng.uniform(-velocity_range, velocity_range,
                                      size=positions.shape)

        personal_best_positions = positions.copy()
        personal_best_scores = np.full(n_particles, np.inf)
        global_best_position = positions[0].copy()
        global_best_score = np.inf

        for iteration in range(n_iterations):
            for i in range(n_particles):
                positions[i] = space.clip(positions[i])
                score = self._evaluate(positions[i]).score
                if score < personal_best_scores[i]:
                    personal_best_scores[i] = score
                    personal_best_positions[i] = positions[i].copy()
                if score < global_best_score:
                    global_best_score = score
                    global_best_position = positions[i].copy()

            r1 = self.rng.random(positions.shape)
            r2 = self.rng.random(positions.shape)
            velocities = params.inertia * (
                velocities
                + params.cognitive * r1 * (personal_best_positions - positions)
                + params.social * r2 * (global_best_position[None, :] - positions)
            )
            velocities = np.clip(velocities, -velocity_range, velocity_range)
            positions = positions + velocities

            self._record(iteration + 1)

### 14.3 GA -- Algoritmo genetico de codificacion real

El unico evolutivo del grupo: en vez de mover agentes en el espacio, construye una nueva
poblacion en cada generacion a partir de la anterior.

1. **Elitismo**: los `elite_count = 2` mejores pasan intactos.
2. **Seleccion por torneo**: se toman `tournament_size = 3` individuos al azar y gana el
   de mejor score. Es una presion selectiva moderada y, sobre todo, **invariante a la
   escala del fitness** -- lo que importa aqui porque los scores penalizados viven en
   escalas muy distintas (~0.28 los factibles, ~10^6 los infactibles) y una seleccion
   proporcional al fitness seria inutilizable.
3. **Cruce BLX-alpha** (probabilidad 0.90): el hijo se muestrea uniforme en el intervalo
   entre padres, **expandido** un `alpha = 0.5` a cada lado. La expansion permite generar
   hijos fuera del rango de los padres, que es lo que evita el colapso prematuro de la
   diversidad.
4. **Mutacion gaussiana**: cada gen muta con probabilidad 0.10, con `sigma` igual al 10%
   del rango de esa variable.

Nota sobre el presupuesto: los individuos de elite se **reevaluan** en la generacion
siguiente, lo que desperdicia unas pocas evaluaciones. Se deja asi a proposito: mantener
el presupuesto identico al de los demas algoritmos (`population x iterations`) importa
mas que ahorrarlas, porque es lo que hace comparable la comparativa.

In [ ]:
class GeneticAlgorithm(Optimizer):
    """GA real-coded: torneo + BLX-alpha + mutacion gaussiana + elitismo."""

    name = "ga"

    def _search(self):
        space = self.space
        params = self.config.ga
        n_pop = self.config.population
        n_iterations = self.config.iterations

        positions = space.sample(self.rng, n_pop)

        for generation in range(n_iterations):
            scores = np.empty(n_pop)
            for i in range(n_pop):
                positions[i] = space.clip(positions[i])
                scores[i] = self._evaluate(positions[i]).score

            positions = self._next_generation(positions, scores, params, space)

            self._record(generation + 1)

    def _next_generation(self, positions, scores, params, space):
        n_pop = len(positions)
        order = np.argsort(scores)
        elite_count = min(params.elite_count, n_pop)

        new_positions = np.empty_like(positions)
        new_positions[:elite_count] = positions[order[:elite_count]]

        for i in range(elite_count, n_pop):
            parent_a = self._tournament_select(positions, scores, params.tournament_size)
            parent_b = self._tournament_select(positions, scores, params.tournament_size)

            if self.rng.random() < params.crossover_rate:
                child = self._crossover(parent_a, parent_b, params.blx_alpha)
            else:
                child = parent_a.copy()

            child = self._mutate(child, params, space)
            new_positions[i] = space.clip(child)

        return new_positions

    def _tournament_select(self, positions, scores, tournament_size):
        """Elige el mejor de ``tournament_size`` individuos tomados al azar."""
        contenders = self.rng.choice(len(positions), size=tournament_size, replace=False)
        winner = contenders[np.argmin(scores[contenders])]
        return positions[winner]

    def _crossover(self, parent_a, parent_b, alpha=0.5):
        """Cruce BLX-alpha: uniforme en el intervalo entre padres, expandido."""
        low = np.minimum(parent_a, parent_b)
        high = np.maximum(parent_a, parent_b)
        span = high - low
        return self.rng.uniform(low - alpha * span, high + alpha * span)

    def _mutate(self, individual, params, space):
        """Mutacion gaussiana gen a gen, con sigma proporcional al rango."""
        sigma = params.mutation_sigma_ratio * (space.upper - space.lower)
        mask = self.rng.random(individual.shape) < params.mutation_rate
        noise = self.rng.normal(0.0, sigma)
        return np.where(mask, individual + noise, individual)

### 14.4 SHO -- Spotted Hyena Optimizer

Estructuralmente es **GWO con un solo lider** en vez de tres: toda la manada se encierra
sobre la "presa", que es la mejor posicion conocida.

$$\vec{D} = |\vec{B} \cdot \vec{X}_{presa} - \vec{X}_i| \qquad
\vec{X}_i \leftarrow \vec{X}_{presa} - \vec{E} \cdot \vec{D}$$

con $\vec{B} = 2\vec{r_1}$ y $\vec{E} = 2h\vec{r_2} - h$.

La diferencia con GWO esta en el coeficiente de exploracion: $h$ decae de
**`h_max = 5.0` a 0**, mientras que el $a$ de GWO decae de 2 a 0. Un rango 2.5 veces mas
ancho da mucha mas exploracion inicial con la misma estructura de encierro -- y tambien
mas varianza entre semillas, que es lo que se observa en los resultados.

**Sobre la fidelidad al paper original.** Esta implementacion sigue la actualizacion de
posicion de la implementacion de referencia de los propios autores (encierro directo
sobre la presa). El paper describe ademas un paso de "formacion de cluster" cuyo tamano
no queda especificado de forma reproducible y que la implementacion de referencia no usa,
asi que no se replica aqui.

In [ ]:
class SpottedHyenaOptimizer(Optimizer):
    """SHO: encierro de toda la manada sobre la presa (mejor posicion conocida)."""

    name = "sho"

    def _search(self):
        space = self.space
        h_max = self.config.sho.h_max
        n_hyenas = self.config.population
        n_iterations = self.config.iterations

        positions = space.sample(self.rng, n_hyenas)
        prey = positions[0].copy()
        prey_score = np.inf

        for iteration in range(n_iterations):
            for i in range(n_hyenas):
                positions[i] = space.clip(positions[i])
                evaluation = self._evaluate(positions[i])
                if evaluation.score < prey_score:
                    prey_score = evaluation.score
                    prey = positions[i].copy()

            # h decae linealmente de h_max a 0: la manada pasa de explorar
            # ampliamente alrededor de la presa a converger sobre ella.
            h = h_max - iteration * (h_max / max(n_iterations - 1, 1))

            positions = self._move(positions, prey, h)

            self._record(iteration + 1, h=h)

    def _move(self, positions, prey, h):
        """Encierro vectorizado de toda la manada sobre la presa."""
        r1 = self.rng.random(positions.shape)
        r2 = self.rng.random(positions.shape)

        B = 2.0 * r1
        E = 2.0 * h * r2 - h

        distance = np.abs(B * prey[None, :] - positions)
        return prey[None, :] - E * distance

### 14.5 WOA -- Whale Optimization Algorithm

Cada ballena sortea, **una vez por iteracion**, entre dos mecanismos con probabilidad
50/50:

**$p < 0.5$ -- encierro.** Igual que GWO y SHO, pero con una vuelta de tuerca: si
$|A| < 1$ se encierra sobre la mejor posicion conocida (explotacion), y si $|A| \ge 1$ se
encierra sobre **una ballena aleatoria de la manada** (exploracion). Es el unico de los
cinco que usa un agente aleatorio como referencia.

**$p \ge 0.5$ -- ataque por burbujas.** Espiral logaritmica alrededor de la mejor
posicion:

$$\vec{X}_i \leftarrow |\vec{X}_{best} - \vec{X}_i| \cdot e^{bl} \cdot \cos(2\pi l)
  + \vec{X}_{best}$$

con $l$ uniforme en $[-1, 1]$ y $b = 1.0$ la constante de la espiral.

$a$ decae linealmente de 2 a 0, igual que en GWO.

**Detalle de implementacion que importa:** igual que en la implementacion de referencia
de los autores, los coeficientes $A$, $C$, $l$ y la moneda $p$ son **escalares por
ballena**, no vectores por dimension. Eso significa que una ballena aplica el mismo tiron
a las cuatro variables, y es lo que impide vectorizar el operador sobre toda la poblacion
como en GWO. El costo extra es irrelevante: la simulacion horaria domina el tiempo, no el
operador de movimiento.

In [ ]:
class WhaleOptimization(Optimizer):
    """WOA: encierro/busqueda aleatoria o espiral bubble-net, sorteados por ballena."""

    name = "woa"

    def _search(self):
        space = self.space
        spiral_constant = self.config.woa.spiral_constant
        n_whales = self.config.population
        n_iterations = self.config.iterations

        positions = space.sample(self.rng, n_whales)
        best = positions[0].copy()
        best_score = np.inf

        for iteration in range(n_iterations):
            for i in range(n_whales):
                positions[i] = space.clip(positions[i])
                evaluation = self._evaluate(positions[i])
                if evaluation.score < best_score:
                    best_score = evaluation.score
                    best = positions[i].copy()

            a = 2.0 - iteration * (2.0 / max(n_iterations - 1, 1))

            positions = self._move(positions, best, a, spiral_constant)

            self._record(iteration + 1, a=a)

    def _move(self, positions, best, a, spiral_constant):
        """Nueva posicion de cada ballena, con coeficientes propios por ballena."""
        n_whales = len(positions)
        new_positions = np.empty_like(positions)

        for i in range(n_whales):
            r1 = self.rng.random()
            r2 = self.rng.random()
            A = 2.0 * a * r1 - a
            C = 2.0 * r2
            p = self.rng.random()

            if p < 0.5:
                if abs(A) < 1.0:
                    # Explotacion: encierro sobre la mejor posicion conocida.
                    distance = np.abs(C * best - positions[i])
                    new_positions[i] = best - A * distance
                else:
                    # Exploracion: encierro sobre una ballena aleatoria de la manada.
                    random_whale = positions[self.rng.integers(0, n_whales)]
                    distance = np.abs(C * random_whale - positions[i])
                    new_positions[i] = random_whale - A * distance
            else:
                # Ataque por burbujas: espiral logaritmica alrededor de la mejor.
                l = self.rng.uniform(-1.0, 1.0)
                distance_to_best = np.abs(best - positions[i])
                new_positions[i] = (
                    distance_to_best * np.exp(spiral_constant * l)
                    * np.cos(2.0 * np.pi * l) + best
                )

        return new_positions

### 14.6 Busqueda aleatoria -- la linea base

Muestrea disenos al azar y se queda con el mejor. Sin memoria, sin poblacion, sin
adaptacion.

Un detalle deliberado la separa de las otras cinco: **muestrea directamente sobre la
malla discreta** -- unidades enteras, potencias multiplo de 5 MW, duraciones del catalogo
-- en lugar de muestrear continuo y proyectar. Asi cada muestra es un diseno valido, y la
comparacion mide **busqueda**, no suerte en el redondeo. Si muestreara continuo, parte de
su desempeno vendria del sesgo que introduce la proyeccion, no del muestreo.

Recibe exactamente el mismo presupuesto que los demas (`population x iterations`
evaluaciones) y reporta su historial cada `population` evaluaciones, para que su curva de
convergencia sea comparable iteracion a iteracion con las de los algoritmos poblacionales.

In [ ]:
class RandomSearch(Optimizer):
    """Muestreo uniforme sobre el espacio discreto, sin memoria."""

    name = "random"

    def _search(self):
        config = self.objective.config
        battery = config.battery
        electrolyzer = config.electrolyzer

        battery_grid = np.arange(battery.power_min_mw,
                                 battery.power_max_mw + 1e-9,
                                 battery.power_step_mw)
        n_durations = len(battery.duration_candidates_h)
        total_capacity = config.constraints.total_generation_capacity_mw

        # Mismo presupuesto que los algoritmos poblacionales, para que la
        # comparacion sea a igual numero de evaluaciones.
        budget = self.config.evaluation_budget
        report_every = max(self.config.population, 1)

        for evaluation_index in range(budget):
            x = np.array([
                self.rng.uniform(0.0, total_capacity),
                self.rng.integers(electrolyzer.min_units, electrolyzer.max_units + 1),
                self.rng.choice(battery_grid),
                self.rng.integers(0, n_durations),
            ], dtype=float)
            self._evaluate(x)

            if (evaluation_index + 1) % report_every == 0:
                self._record((evaluation_index + 1) // report_every)

        # Garantiza una fila final aunque el presupuesto no sea multiplo exacto.
        if not self._history or self._history[-1]["n_evaluations"] < budget:
            self._record(len(self._history) + 1)


REGISTRY = {
    GreyWolfOptimizer.name: GreyWolfOptimizer,
    ParticleSwarm.name: ParticleSwarm,
    GeneticAlgorithm.name: GeneticAlgorithm,
    SpottedHyenaOptimizer.name: SpottedHyenaOptimizer,
    WhaleOptimization.name: WhaleOptimization,
    RandomSearch.name: RandomSearch,
}


def get_optimizer(name):
    """Busca un optimizador por nombre, con un error util si no existe."""
    try:
        return REGISTRY[name]
    except KeyError:
        raise KeyError("algoritmo desconocido '{}'. Disponibles: {}".format(
            name, ", ".join(sorted(REGISTRY)))) from None


# Tabla de hiperparametros efectivamente en uso, leida de la configuracion.
_mh = ESCENARIO.metaheuristic
tabla_hiperparametros = pd.DataFrame([
    ["gwo", "Grey Wolf Optimizer", "enjambre jerarquico",
     "a decae 2 -> 0; lideres alfa, beta, delta", "Mirjalili et al. (2014)"],
    ["pso", "Particle Swarm Optimization", "enjambre",
     "chi={}; c1=c2={}; |v| <= {:.0%} del rango".format(
         _mh.pso.inertia, _mh.pso.cognitive, _mh.pso.velocity_clamp_ratio),
     "Clerc & Kennedy (2002)"],
    ["ga", "Genetic Algorithm", "evolutivo",
     "torneo k={}; BLX-{} p={}; mutacion p={} sigma={:.0%}; elite={}".format(
         _mh.ga.tournament_size, _mh.ga.blx_alpha, _mh.ga.crossover_rate,
         _mh.ga.mutation_rate, _mh.ga.mutation_sigma_ratio, _mh.ga.elite_count),
     "real-coded GA estandar"],
    ["sho", "Spotted Hyena Optimizer", "enjambre jerarquico",
     "h decae {} -> 0; lider unico (presa)".format(_mh.sho.h_max),
     "Dhiman & Kumar (2017)"],
    ["woa", "Whale Optimization Algorithm", "enjambre",
     "encierro/espiral 50/50; b={}; a decae 2 -> 0".format(_mh.woa.spiral_constant),
     "Mirjalili & Lewis (2016)"],
    ["random", "Random Search", "linea base",
     "muestreo uniforme sobre la malla discreta", "--"],
], columns=["codigo", "nombre", "familia", "hiperparametros", "fuente"])

display(tabla_hiperparametros)
print("Presupuesto identico para los {} algoritmos: {} evaluaciones por corrida "
      "({} individuos x {} iteraciones)".format(
          len(REGISTRY), _mh.evaluation_budget, _mh.population, _mh.iterations))
print("Penalizacion de infactibles: {:.0e} + {:.0e} * exceso_de_AGSR".format(
    _mh.penalty_infeasible, _mh.penalty_agsr_weight))

## 15. Identidad visual de los algoritmos

Cada algoritmo tiene **un color fijo en todas las figuras**, asignado por identidad y
nunca por rango dentro de un grafico. Si `gwo` fuera azul en una figura y verde en la
siguiente, el lector perderia la referencia.

Dos decisiones deliberadas:

- **`random` no recibe un matiz categorico.** Es la linea base, no un competidor, asi que
  va en gris neutro con linea discontinua y marcador "x" -- la convencion habitual para
  una referencia.
- **En el grafico de dispersion, la forma del marcador es la codificacion primaria.** Con
  5 series categoricas, el color solo no alcanza en el contexto "todos los pares visibles
  a la vez": la distancia percibida entre el magenta y el naranja cae por debajo del
  umbral de discriminacion. Por eso ahi cada algoritmo tiene ademas su propia forma.

In [ ]:
ALGORITHM_ORDER = ("gwo", "pso", "ga", "sho", "woa", "random")

ALGORITHM_COLORS = {
    "gwo": "#2a78d6",      # azul
    "pso": "#eb6834",      # naranja
    "ga": "#1baf7a",       # verde-agua
    "sho": "#eda100",      # amarillo
    "woa": "#e87ba4",      # magenta
    "random": "#898781",   # gris neutro -- linea base, no una serie categorica
}

ALGORITHM_LINESTYLES = {
    "gwo": "-", "pso": "-", "ga": "-", "sho": "-", "woa": "-", "random": "--",
}

# La codificacion primaria de identidad en los graficos de dispersion.
ALGORITHM_MARKERS = {
    "gwo": "o", "pso": "s", "ga": "^", "sho": "D", "woa": "v", "random": "x",
}


def algorithm_color(name):
    return ALGORITHM_COLORS.get(name, "#666666")


def algorithm_linestyle(name):
    return ALGORITHM_LINESTYLES.get(name, "-")


def algorithm_marker(name):
    return ALGORITHM_MARKERS.get(name, "o")


fig, ax = plt.subplots(figsize=(9, 1.7))
for i, nombre in enumerate(ALGORITHM_ORDER):
    marcador = algorithm_marker(nombre)
    # "x" es un marcador sin relleno: matplotlib ignora edgecolor y avisa.
    extra = ({"linewidths": 2.2} if marcador == "x"
             else {"linewidths": 0.6, "edgecolor": "white"})
    ax.scatter([i], [0], s=220, color=algorithm_color(nombre), marker=marcador,
               **extra)
    ax.text(i, -0.45, nombre.upper(), ha="center", va="top", fontsize=10)
ax.set_xlim(-0.6, len(ALGORITHM_ORDER) - 0.4)
ax.set_ylim(-1.1, 0.6)
ax.axis("off")
ax.set_title("Color y marcador por algoritmo", fontsize=10)
plt.tight_layout()
plt.show()

## 16. Ejecucion de la comparativa

Un solo bucle `algoritmo x semilla`. Los 6 algoritmos comparten:

- **el mismo objeto `ObjectiveFunction`**, y por lo tanto el mismo cache de perfiles de
  generacion -- ninguno se beneficia de un modelo ligeramente distinto;
- **las mismas semillas**, de modo que la comparacion posterior puede ser **pareada**:
  la semilla 42 define el mismo punto de partida y el mismo orden de muestreo para todos;
- **el mismo presupuesto** de evaluaciones.

Una sola corrida de una metaheuristica estocastica no es un resultado publicable. Lo que
se reporta es la **distribucion sobre semillas**, y de ahi que el modo completo corra 30.

> Esta es la celda lenta del notebook. En modo `"rapido"` toma unos segundos; en
> `"completo"`, alrededor de 12 minutos.

In [ ]:
def run_comparison(objective, algorithms, seeds, on_result=None):
    """Corre cada algoritmo sobre cada semilla y devuelve ``(history, runs)``."""
    histories = []
    summaries = []

    for algorithm in algorithms:
        optimizer_class = get_optimizer(algorithm)
        for seed in seeds:
            result = optimizer_class(objective, seed=seed).optimize()
            history = result.history.copy()
            history["algorithm"] = algorithm
            history["seed"] = seed
            histories.append(history)
            summaries.append(result.summary())
            if on_result is not None:
                on_result(algorithm, seed, result)

    return pd.concat(histories, ignore_index=True), pd.DataFrame(summaries)


objetivo = ObjectiveFunction(hourly, ESCENARIO)


def _progreso(algorithm, seed, result):
    design = result.best.design
    print("{:>6} semilla={:<3} score={:.6f}  W={:5.1f} PV={:5.1f} E={:3.0f} "
          "B={:2.0f}/{:.0f}h  factible={:<5} {:5.1f}s".format(
              algorithm, seed, result.best_score, design.wind_mw, design.pv_mw,
              design.electrolyzer_mw, design.battery_mw, design.battery_duration_h,
              str(result.best.result.feasible), result.elapsed_s))


total_sims = len(ALGORITMOS) * len(SEMILLAS) * ESCENARIO.metaheuristic.evaluation_budget
print("Corriendo {} algoritmos x {} semillas x {} evaluaciones = {:,} simulaciones "
      "horarias...".format(len(ALGORITMOS), len(SEMILLAS),
                           ESCENARIO.metaheuristic.evaluation_budget, total_sims))
print()

t0 = time.perf_counter()
history, runs = run_comparison(objetivo, ALGORITMOS, SEMILLAS, on_result=_progreso)
elapsed_total = time.perf_counter() - t0

print()
print("Comparativa completa en {:.1f} s ({:.1f} min)".format(
    elapsed_total, elapsed_total / 60.0))
print("Cache de perfiles:", objetivo.cache.stats())

## 17. Resultados de la comparativa

El ranking se ordena por **mejor score alcanzado** (menor LCOE penalizado es mejor), pero
las columnas que importan para juzgar un optimizador estocastico son otras:

- **`mean` y `median`** dicen que se puede esperar de una corrida cualquiera.
- **`std`** dice cuanto depende el resultado de la suerte de la semilla. Un algoritmo con
  el mejor `best` pero `std` alta es peor en la practica que uno consistente: en
  produccion se corre una vez, no treinta.
- **`feasible_runs`** dice cuantas corridas terminaron en una solucion que respeta el
  AGSR. Si no son todas, el ranking por score no significa nada.
- **`mean_elapsed_s`** es el costo. Con presupuesto de evaluaciones identico, las
  diferencias de tiempo vienen del operador de movimiento, no de la busqueda.

In [ ]:
def aggregate_runs(runs):
    """Estadistica sobre varias semillas de un mismo algoritmo.

    Una sola corrida de una metaheuristica estocastica no es un resultado publicable;
    lo que se reporta es la distribucion sobre semillas.
    """
    frame = pd.DataFrame(runs)
    if frame.empty:
        return frame

    stats_table = frame.groupby("algorithm")["score"].agg(
        runs="count", best="min", mean="mean", std="std", worst="max", median="median")
    stats_table["feasible_runs"] = frame.groupby("algorithm")["feasible"].sum()
    stats_table["mean_elapsed_s"] = frame.groupby("algorithm")["elapsed_s"].mean()
    return stats_table.reset_index()


def comparison_table(runs):
    """Estadistica descriptiva por algoritmo, con ranking por mejor score."""
    stats_table = aggregate_runs(runs)
    if stats_table.empty:
        return stats_table
    stats_table = stats_table.sort_values(["best", "mean"]).reset_index(drop=True)
    stats_table.insert(0, "ranking", range(1, len(stats_table) + 1))
    return stats_table


ranking = comparison_table(runs)
display(ranking)

mejor = ranking.iloc[0]
mas_estable = ranking.sort_values("std").iloc[0]
peor = ranking.iloc[-1]

display(Markdown(
    "- **Mejor score absoluto:** `{}` con {:.6f}.\n"
    "- **Mas consistente entre semillas:** `{}` (desviacion {:.6f}).\n"
    "- **Peor score:** `{}` con {:.6f}.\n"
    "- Corridas factibles: {} de {} en total.".format(
        mejor["algorithm"], mejor["best"],
        mas_estable["algorithm"], mas_estable["std"],
        peor["algorithm"], peor["best"],
        int(ranking["feasible_runs"].sum()), int(ranking["runs"].sum()))
))

# El mejor diseno encontrado por cualquier algoritmo, en cualquier semilla.
mejor_corrida = runs.sort_values("score").iloc[0]
display(Markdown(
    "**Mejor diseno discreto encontrado** (`{}`, semilla {}): "
    "W={:.1f} MW, PV={:.1f} MW, E={:.0f} MW ({} unidades), "
    "B={:.0f} MW / {:.0f} h = {:.0f} MWh -- "
    "LCOE **{:.6f} CNY/kWh**, LCOH {:.2f} CNY/kg, AGSR {:.4f}, "
    "CF electrolizador {:.4f}.".format(
        mejor_corrida["algorithm"], int(mejor_corrida["seed"]),
        mejor_corrida["wind_mw"], mejor_corrida["pv_mw"],
        mejor_corrida["electrolyzer_mw"], int(mejor_corrida["n_electrolyzer_units"]),
        mejor_corrida["battery_mw"], mejor_corrida["battery_duration_h"],
        mejor_corrida["battery_mwh"], mejor_corrida["lcoe_cny_per_kwh"],
        mejor_corrida["lcoh_cny_per_kg"], mejor_corrida["agsr"],
        mejor_corrida["electrolyzer_cf"])
))

### 17.1 Convergencia

Se grafica la **mediana** del mejor score entre semillas, con la banda intercuartil
(percentiles 25-75) sombreada. La mediana y no la media porque una sola semilla que quede
atrapada en la region infactible (score ~10^6) arrastraria la media fuera de escala; la
banda porque una curva de convergencia sin dispersion oculta justamente lo que distingue
a un algoritmo confiable de uno afortunado.

El eje de las abscisas son iteraciones, que con presupuesto identico equivalen a
evaluaciones: los 6 algoritmos han visto el mismo numero de disenos en cada punto del eje.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))

for nombre in ALGORITHM_ORDER:
    grupo = history[history["algorithm"] == nombre]
    if grupo.empty:
        continue
    resumen = (grupo.groupby("iteration")["best_score"]
               .agg(mediana="median",
                    q25=lambda s: s.quantile(0.25),
                    q75=lambda s: s.quantile(0.75))
               .reset_index())
    color = algorithm_color(nombre)
    ax.plot(resumen["iteration"], resumen["mediana"], label=nombre.upper(),
            color=color, linestyle=algorithm_linestyle(nombre), lw=1.9)
    ax.fill_between(resumen["iteration"], resumen["q25"], resumen["q75"],
                    color=color, alpha=0.13, linewidth=0)

ax.set_xlabel("Iteracion")
ax.set_ylabel("Mejor score (LCOE penalizado) [CNY/kWh]")
ax.set_title("Convergencia por algoritmo -- mediana y rango intercuartil sobre {} semillas"
             .format(len(SEMILLAS)))
ax.legend(ncol=3)

# La escala util es la de la region factible: si alguna semilla arranca infactible
# (score ~1e6) el eje completo queda inservible.
factibles = history.loc[history["best_score"] < 1.0, "best_score"]
if not factibles.empty:
    ax.set_ylim(factibles.min() * 0.995, factibles.quantile(0.995) * 1.02)

plt.tight_layout()
plt.show()

### 17.2 Distribucion de los scores finales y costo de computo

El **boxplot** muestra la dispersion completa: caja intercuartil, mediana, bigotes y
valores atipicos. Es la lectura honesta de "que pasa si corro este algoritmo una vez".

La **dispersion calidad-vs-tiempo** cruza lo que se obtiene con lo que cuesta. Con el
presupuesto de evaluaciones fijado, un algoritmo que tarda mas esta gastando ese tiempo
en su operador de movimiento, no en explorar mas: el eje horizontal mide sobrecarga
algoritmica, no esfuerzo de busqueda.

In [ ]:
orden = [a for a in ranking["algorithm"].tolist()]

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.5))

# --- Boxplot de scores finales ------------------------------------------------
datos = [runs.loc[runs["algorithm"] == a, "score"].to_numpy() for a in orden]
caja = axes[0].boxplot(datos, tick_labels=[a.upper() for a in orden],
                       patch_artist=True, medianprops={"color": "#222222", "lw": 1.6})
for parche, nombre in zip(caja["boxes"], orden):
    parche.set_facecolor(algorithm_color(nombre))
    parche.set_alpha(0.55)
    parche.set_edgecolor("#444444")
axes[0].set_xlabel("Algoritmo (ordenado por ranking)")
axes[0].set_ylabel("Score final [CNY/kWh]")
axes[0].set_title("Distribucion del score final")

# --- Calidad vs tiempo --------------------------------------------------------
for nombre in ALGORITHM_ORDER:
    grupo = runs[runs["algorithm"] == nombre]
    if grupo.empty:
        continue
    marcador = algorithm_marker(nombre)
    # "x" es un marcador sin relleno: no admite edgecolor.
    extra = ({"linewidths": 2.0} if marcador == "x"
             else {"linewidths": 0.6, "edgecolor": "white"})
    axes[1].scatter(grupo["elapsed_s"], grupo["score"],
                    label=nombre.upper(), color=algorithm_color(nombre),
                    marker=marcador, s=62, alpha=0.85, **extra)
axes[1].set_xlabel("Tiempo de la corrida [s]")
axes[1].set_ylabel("Score final [CNY/kWh]")
axes[1].set_title("Calidad alcanzada vs. costo de computo\n"
                  "(una marca por corrida; la forma identifica el algoritmo)")
axes[1].legend(ncol=2, fontsize=9)

plt.tight_layout()
plt.show()

## 18. Significancia estadistica

"El algoritmo A dio mejor media" **no es una conclusion defendible** por si sola: podria
ser ruido de semilla. Hacen falta dos cosas mas, y las dos estan aqui.

### Test de hipotesis: Wilcoxon pareado

- **Pareado por semilla** porque la misma semilla define el mismo punto de partida y el
  mismo orden de muestreo para todos los algoritmos. Comparar semilla a semilla
  **controla** esa fuente de varianza en vez de promediarla.
- **No parametrico** porque los scores de un optimizador estocastico no son normales en
  general -- estan acotados por abajo por el optimo y tienen cola larga hacia arriba.

### Correccion por comparaciones multiples: Holm-Bonferroni

Con 6 algoritmos hay **15 pares**. Sin corregir, la probabilidad de encontrar al menos un
"significativo" por puro azar entre los 15 supera el 50%, no el 5% nominal.
Holm-Bonferroni ordena los p-valores de menor a mayor y multiplica el i-esimo por
$(m - i)$, garantizando monotonia. Es menos conservador que Bonferroni simple y controla
igualmente la tasa de error familiar.

### Tamano del efecto: A12 de Vargha-Delaney

Un p-valor dice *si* hay diferencia, no *cuanta*. A12 es la probabilidad de que una
corrida al azar del algoritmo A obtenga mejor score que una de B, mas la mitad de la
probabilidad de empate:

- **0.5** = indistinguibles
- **1.0** = A domina siempre a B
- **0.0** = B domina siempre a A

Es el tamano del efecto estandar en la literatura de metaheuristicas (Arcuri & Briand,
2011) precisamente porque no asume normalidad ni varianzas iguales.

In [ ]:
def vargha_delaney_a12(a, b):
    """Tamano del efecto A12 de Vargha-Delaney.

    Probabilidad de que una corrida tomada al azar de ``a`` obtenga un score mejor
    (menor, porque el objetivo es LCOE) que una de ``b``, mas la mitad de la
    probabilidad de empate.
    """
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    n_a, n_b = len(a), len(b)
    if n_a == 0 or n_b == 0:
        raise ValueError("ambas muestras deben tener al menos una corrida")

    wins = 0.0
    for value in a:
        wins += np.sum(value < b) + 0.5 * np.sum(value == b)
    return wins / (n_a * n_b)


def _holm_bonferroni(p_values):
    """Correccion de Holm-Bonferroni: menos conservadora que Bonferroni simple.

    Ordena los p-valores de menor a mayor y multiplica el i-esimo por (m - i),
    garantizando monotonia antes de recortar a 1.0.
    """
    m = len(p_values)
    order = np.argsort(p_values)
    adjusted = np.empty(m)
    running_max = 0.0
    for rank, index in enumerate(order):
        corrected = min((m - rank) * p_values[index], 1.0)
        running_max = max(running_max, corrected)
        adjusted[index] = running_max
    return adjusted


def pairwise_wilcoxon(runs, score_column="score"):
    """Wilcoxon pareado por semilla entre cada par de algoritmos."""
    algorithms = sorted(runs["algorithm"].unique())
    if len(algorithms) < 2:
        raise ValueError("hacen falta al menos dos algoritmos para comparar")

    pivot = runs.pivot(index="seed", columns="algorithm", values=score_column)
    pivot = pivot.dropna()   # solo semillas presentes en todos los algoritmos
    if len(pivot) < 2:
        raise ValueError(
            "hacen falta al menos dos semillas comunes a todos los algoritmos para "
            "el test pareado; hay {}".format(len(pivot)))

    rows = []
    for algo_a, algo_b in combinations(algorithms, 2):
        sample_a = pivot[algo_a].to_numpy()
        sample_b = pivot[algo_b].to_numpy()
        differences = sample_a - sample_b

        if np.allclose(differences, 0.0):
            # scipy.stats.wilcoxon lanza ValueError si todas las diferencias son
            # cero; es un resultado legitimo, no un error de calculo.
            p_value = 1.0
        else:
            _, p_value = stats.wilcoxon(sample_a, sample_b)

        rows.append({
            "algoritmo_a": algo_a, "algoritmo_b": algo_b,
            "mediana_a": float(np.median(sample_a)),
            "mediana_b": float(np.median(sample_b)),
            "p_value": float(p_value),
            "a12": vargha_delaney_a12(sample_a, sample_b),
            "n_semillas": len(pivot),
        })

    table = pd.DataFrame(rows)
    table["p_value_holm"] = _holm_bonferroni(table["p_value"].to_numpy())
    table["significativo_alpha_0.05"] = table["p_value_holm"] < 0.05
    return table


if len(SEMILLAS) >= 2:
    tests = pairwise_wilcoxon(runs)
    display(tests[["algoritmo_a", "algoritmo_b", "mediana_a", "mediana_b",
                   "p_value", "p_value_holm", "a12", "significativo_alpha_0.05"]])

    n_signif = int(tests["significativo_alpha_0.05"].sum())
    display(Markdown(
        "**{} de {} pares** muestran una diferencia significativa a alpha = 0.05 tras la "
        "correccion de Holm-Bonferroni, sobre {} semillas pareadas.".format(
            n_signif, len(tests), int(tests["n_semillas"].iloc[0]))
    ))
    if n_signif == 0:
        display(Markdown(
            "> Con este presupuesto no hay diferencias estadisticamente distinguibles. "
            "Eso **es** un resultado: significa que, para este problema y estas "
            "{} semillas, elegir entre estos algoritmos no cambia el diseno obtenido. "
            "Con `MODO = \"completo\"` (30 semillas) la potencia del test aumenta y "
            "algunos pares si se separan.".format(len(SEMILLAS))
        ))
else:
    tests = pd.DataFrame()
    print("El test pareado necesita al menos 2 semillas; hay", len(SEMILLAS))

In [ ]:
if not tests.empty:
    # Mapa de calor del tamano del efecto: cuanto domina la fila a la columna.
    algoritmos = sorted(runs["algorithm"].unique())
    matriz = pd.DataFrame(np.full((len(algoritmos), len(algoritmos)), np.nan),
                          index=algoritmos, columns=algoritmos)
    for _, fila in tests.iterrows():
        matriz.loc[fila["algoritmo_a"], fila["algoritmo_b"]] = fila["a12"]
        matriz.loc[fila["algoritmo_b"], fila["algoritmo_a"]] = 1.0 - fila["a12"]

    fig, ax = plt.subplots(figsize=(7.2, 6))
    im = ax.imshow(matriz.to_numpy(), cmap="RdBu", vmin=0.0, vmax=1.0)
    ax.set_xticks(range(len(algoritmos)), [a.upper() for a in algoritmos])
    ax.set_yticks(range(len(algoritmos)), [a.upper() for a in algoritmos])
    ax.set_xlabel("... frente a este")
    ax.set_ylabel("Este algoritmo ...")
    ax.set_title("Tamano del efecto A12\n"
                 "(> 0.5: la fila gana mas seguido que la columna)")
    ax.grid(False)

    for i, fila_nombre in enumerate(algoritmos):
        for j, col_nombre in enumerate(algoritmos):
            valor = matriz.iloc[i, j]
            if np.isfinite(valor):
                ax.text(j, i, "{:.2f}".format(valor), ha="center", va="center",
                        fontsize=9,
                        color="white" if abs(valor - 0.5) > 0.28 else "#222222")
            else:
                ax.text(j, i, "--", ha="center", va="center", color="#999999")

    fig.colorbar(im, ax=ax, label="A12", fraction=0.046)
    plt.tight_layout()
    plt.show()

## 19. Sensibilidad

Las dos restricciones que definen el dominio factible se barren una a la vez, desde el
escenario de replicacion, usando el **barrido exhaustivo** de la seccion 11 y no una
metaheuristica. La razon es metodologica: el barrido es determinista, de modo que las
curvas de sensibilidad no llevan ruido de semilla y la tendencia se lee sin ambiguedad.

**Limite de AGSR** -- es la restriccion que define el dominio factible y, segun §3.1 del
paper, la que fija el limite inferior del electrolizador. Relajarla deberia agrandar el
dominio y bajar el LCOE optimo.

**Carga minima del electrolizador** -- el paper usa 30%. Es lo que determina el
dimensionamiento de la bateria: cuanto mas alto el minimo, mas horas quedan por debajo y
mas bateria hace falta para sostener la operacion.

Cada valor barrido son 420 evaluaciones del modelo base, unos 3 segundos.

In [ ]:
AGSR_MAX_VALUES = (0.10, 0.15, 0.20, 0.25, 0.30)
MIN_LOAD_RATIO_VALUES = (0.20, 0.30, 0.40, 0.50)

_SUMMARY_KEYS = ("evaluated", "feasible", "infeasible", "best_lcoe_cny_per_kwh",
                 "best_wind_mw", "best_electrolyzer_mw", "best_electrolyzer_cf")


def _sweep(hourly, config, parameter, values, build_config):
    rows = []
    for value in values:
        swept_config = build_config(config, value)
        results = run_grid_search(hourly, swept_config)
        summary = summarize_grid(results)

        row = {"parameter": parameter, "value": float(value)}
        for key in _SUMMARY_KEYS:
            row[key] = summary.get(key, float("nan"))
        row["feasible_share"] = (summary["feasible"] / summary["evaluated"]
                                 if summary["evaluated"] else 0.0)
        rows.append(row)
    return pd.DataFrame(rows)


def sweep_agsr_max(hourly, config, values=AGSR_MAX_VALUES):
    """Optimo y tamano del dominio factible en funcion del limite de AGSR."""
    def build_config(base, value):
        return base.replace(constraints=replace(base.constraints, agsr_max=value))
    return _sweep(hourly, config, "agsr_max", values, build_config)


def sweep_min_load_ratio(hourly, config, values=MIN_LOAD_RATIO_VALUES):
    """Optimo y tamano del dominio factible en funcion de la carga minima."""
    def build_config(base, value):
        return base.replace(electrolyzer=replace(base.electrolyzer,
                                                 min_load_ratio=value))
    return _sweep(hourly, config, "min_load_ratio", values, build_config)


t0 = time.perf_counter()
sens_agsr = sweep_agsr_max(hourly, ESCENARIO)
sens_carga = sweep_min_load_ratio(hourly, ESCENARIO)
print("Sensibilidad completa en {:.1f} s ({} barridos de {} configuraciones)".format(
    time.perf_counter() - t0,
    len(AGSR_MAX_VALUES) + len(MIN_LOAD_RATIO_VALUES),
    int(sens_agsr["evaluated"].iloc[0])))

print()
print("Limite de AGSR:")
display(sens_agsr[["value", "feasible", "feasible_share", "best_lcoe_cny_per_kwh",
                   "best_wind_mw", "best_electrolyzer_mw", "best_electrolyzer_cf"]])

print("Carga minima del electrolizador:")
display(sens_carga[["value", "feasible", "feasible_share", "best_lcoe_cny_per_kwh",
                    "best_wind_mw", "best_electrolyzer_mw", "best_electrolyzer_cf"]])

In [ ]:
def plot_sensitivity_sweep(sweep, etiqueta):
    """Tres paneles de un eje cada uno: costo, capacidades y factibilidad.

    Ningun panel usa eje Y doble: donde hay mas de una magnitud se resuelve en
    paneles separados, cada uno con una sola escala.
    """
    datos = sweep.sort_values("value")
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))

    axes[0].plot(datos["value"], datos["best_lcoe_cny_per_kwh"], marker="o",
                 color="#2a78d6", lw=2)
    axes[0].set_ylabel("Mejor LCOE [CNY/kWh]")
    axes[0].set_title("Costo del optimo")

    axes[1].plot(datos["value"], datos["best_wind_mw"], marker="o",
                 color="#1baf7a", lw=2, label="Eolica [MW]")
    axes[1].plot(datos["value"], datos["best_electrolyzer_mw"], marker="s",
                 color="#eb6834", lw=2, label="Electrolizador [MW]")
    axes[1].set_ylabel("Capacidad [MW]")
    axes[1].set_title("Capacidades del optimo")
    axes[1].legend()

    axes[2].plot(datos["value"], datos["feasible_share"], marker="o",
                 color="#eda100", lw=2)
    axes[2].set_ylabel("Fraccion factible del dominio")
    axes[2].set_title("Tamano del dominio factible")

    for ax in axes:
        ax.set_xlabel(etiqueta)

    fig.suptitle("Sensibilidad a: {}".format(etiqueta))
    plt.tight_layout()
    return fig


plot_sensitivity_sweep(sens_agsr, "Limite de AGSR")
plt.show()

plot_sensitivity_sweep(sens_carga, "Carga minima del electrolizador")
plt.show()

In [ ]:
# Lectura cuantitativa de los dos barridos.
a_lo, a_hi = sens_agsr.iloc[0], sens_agsr.iloc[-1]
c_lo, c_hi = sens_carga.iloc[0], sens_carga.iloc[-1]

display(Markdown(
    "- **Limite de AGSR:** de {:.0%} a {:.0%}, el dominio factible pasa de {} a {} "
    "configuraciones ({:+d}) y el LCOE optimo {} en {:.4f} CNY/kWh. Coherente con que "
    "el paper (§3.1) fije el limite inferior del electrolizador precisamente por esta "
    "restriccion: si se puede vender mas a la red, hace falta menos electrolisis.\n"
    "- **Carga minima:** de {:.0%} a {:.0%} (el paper usa {:.0%}), el dominio factible "
    "cambia en {:+d} configuraciones y el LCOE optimo {} en {:.4f} CNY/kWh.".format(
        a_lo["value"], a_hi["value"], int(a_lo["feasible"]), int(a_hi["feasible"]),
        int(a_hi["feasible"] - a_lo["feasible"]),
        "baja" if a_hi["best_lcoe_cny_per_kwh"] < a_lo["best_lcoe_cny_per_kwh"] else "sube",
        abs(a_hi["best_lcoe_cny_per_kwh"] - a_lo["best_lcoe_cny_per_kwh"]),
        c_lo["value"], c_hi["value"], ESCENARIO.electrolyzer.min_load_ratio,
        int(c_hi["feasible"] - c_lo["feasible"]),
        "sube" if c_hi["best_lcoe_cny_per_kwh"] > c_lo["best_lcoe_cny_per_kwh"] else "baja",
        abs(c_hi["best_lcoe_cny_per_kwh"] - c_lo["best_lcoe_cny_per_kwh"]))
))

## 20. Lectura de resultados

La celda siguiente arma el resumen con los numeros de **esta** corrida, no con valores
fijos: si se cambia `MODO` o cualquier parametro del escenario, el texto se recalcula.

In [ ]:
lineas = []

lineas.append("### Datos y modelo\n")
lineas.append(
    "- El ano meteorologico usado tiene **{} horas**. La diferencia respecto de 8760 se "
    "explica porque 2008 fue bisiesto.".format(len(hourly)))
lineas.append(
    "- El recurso replicado da un CF eolico de **{:.4f}** (paper: 0.40) y un CF "
    "fotovoltaico de **{:.4f}** (paper: 0.19).".format(cf_wind, cf_pv))
lineas.append(
    "- La validacion contra el paper deja **{} de {} objetivos dentro de tolerancia**."
    .format(n_ok, n_con_dato))

lineas.append("\n### Instancia base (barrido exhaustivo)\n")
lineas.append(
    "- Se evaluaron **{} configuraciones**, de las cuales **{}** ({:.1%}) respetan el "
    "limite de AGSR.".format(resumen_base["evaluated"], resumen_base["feasible"],
                             resumen_base["feasible"] / resumen_base["evaluated"]))
lineas.append(
    "- El optimo global por enumeracion es **W={:.0f} / PV={:.0f} / E={:.0f} / "
    "B={:.1f} MW**, con LCOE **{:.4f} CNY/kWh** y AGSR {:.4f}.".format(
        base_best.wind_mw, base_best.pv_mw, base_best.electrolyzer_mw,
        base_best.battery_mw, base_best.lcoe_cny_per_kwh, base_best.agsr))
lineas.append(
    "- Al ser un dominio enumerable, este valor no es \"el mejor encontrado\" sino **el "
    "mejor que existe** en esa malla: sirve de referencia dura.")

lineas.append("\n### Instancia discreta (metaheuristicas)\n")
lineas.append(
    "- {} algoritmos x {} semillas x {} evaluaciones = **{:,} simulaciones horarias**, "
    "en {:.1f} minutos.".format(len(ALGORITMOS), len(SEMILLAS),
                                ESCENARIO.metaheuristic.evaluation_budget,
                                total_sims, elapsed_total / 60.0))
lineas.append(
    "- Ranking por mejor score: {}.".format(
        ", ".join("**{}** ({:.6f})".format(r["algorithm"], r["best"])
                  for _, r in ranking.iterrows())))
lineas.append(
    "- El mejor diseno hallado es **W={:.1f} / PV={:.1f} / E={:.0f} MW ({} unidades) / "
    "B={:.0f} MW x {:.0f} h**, con LCOE **{:.6f} CNY/kWh** y LCOH {:.2f} CNY/kg.".format(
        mejor_corrida["wind_mw"], mejor_corrida["pv_mw"],
        mejor_corrida["electrolyzer_mw"], int(mejor_corrida["n_electrolyzer_units"]),
        mejor_corrida["battery_mw"], mejor_corrida["battery_duration_h"],
        mejor_corrida["lcoe_cny_per_kwh"], mejor_corrida["lcoh_cny_per_kg"]))
lineas.append(
    "- El algoritmo mas **consistente** entre semillas es `{}` (desviacion {:.6f}); "
    "el mas disperso es `{}` ({:.6f}). Para una corrida unica -- que es lo que se hace "
    "en la practica -- la consistencia pesa mas que el mejor caso.".format(
        mas_estable["algorithm"], mas_estable["std"],
        ranking.sort_values("std").iloc[-1]["algorithm"],
        ranking.sort_values("std").iloc[-1]["std"]))

if not tests.empty:
    n_signif = int(tests["significativo_alpha_0.05"].sum())
    lineas.append(
        "- **{} de {} pares** son estadisticamente distinguibles a alpha=0.05 tras "
        "Holm-Bonferroni. El resto de las diferencias de media no se puede separar del "
        "ruido de semilla con {} corridas.".format(n_signif, len(tests), len(SEMILLAS)))
    linea_base = ranking[ranking["algorithm"] == "random"]
    if not linea_base.empty:
        pos = int(linea_base.iloc[0]["ranking"])
        lineas.append(
            "- La **linea base aleatoria** quedo en la posicion {} de {}. {}".format(
                pos, len(ranking),
                "Las metaheuristicas que quedan por debajo no estan justificando su "
                "complejidad en este problema."
                if pos < len(ranking) else
                "Todas las metaheuristicas la superan, que es la condicion minima para "
                "que valga la pena usarlas."))

lineas.append("\n### Sensibilidad\n")
lineas.append(
    "- El limite de AGSR es la restriccion dominante: moverlo de {:.0%} a {:.0%} cambia "
    "el LCOE optimo en {:.4f} CNY/kWh.".format(
        a_lo["value"], a_hi["value"],
        abs(a_hi["best_lcoe_cny_per_kwh"] - a_lo["best_lcoe_cny_per_kwh"])))
lineas.append(
    "- La carga minima del electrolizador tiene un efecto un orden de magnitud menor "
    "({:.4f} CNY/kWh entre {:.0%} y {:.0%}), pero recorta el dominio factible.".format(
        abs(c_hi["best_lcoe_cny_per_kwh"] - c_lo["best_lcoe_cny_per_kwh"]),
        c_lo["value"], c_hi["value"]))

if MODO == "rapido":
    lineas.append(
        "\n> **Nota:** esta corrida usa el presupuesto reducido "
        "(`MODO = \"rapido\"`: {} semillas x {} individuos x {} iteraciones). Los "
        "numeros de la comparativa y de los tests son indicativos. Para el experimento "
        "reportado, cambiar a `MODO = \"completo\"` en la seccion 1.".format(
            len(SEMILLAS), ESCENARIO.metaheuristic.population,
            ESCENARIO.metaheuristic.iterations))

display(Markdown("\n".join(lineas)))

## 21. Supuestos declarados y limitaciones

Todo lo que sigue es una decision de modelado tomada donde el paper no publica el dato.
Se declara para que el resultado se pueda auditar, no para justificarlo.

1. **Costo de bateria por potencia y energia (30/70).** El paper solo cotiza la bateria
   por kW porque sus tres casos publicados son todos de 1 h. El reparto es NPC-neutro a
   1 h y hace que alargar la bateria cueste. Sin el, la duracion no seria interpretable
   como variable de decision (seccion 4).
2. **`wind_speed_source = "measured_50m"`.** La Tabla 3 declara buje a 105 m, pero
   extrapolar a esa altura da un CF eolico de ~56% contra el 40% publicado, mientras que
   la serie de 50 m sin extrapolar lo reproduce al 1%. El modelo del paper debe incluir
   perdidas que no publica.
3. **`plant_load_ratio = 0.0`.** El paper no publica el consumo parasito de la planta.
4. **Hiperparametros sin ajustar.** Valores canonicos de la literatura para PSO, GA, SHO
   y WOA. La comparativa no es un torneo con un favorito afinado.
5. **NPC W60-P140-E85-B27.5.** Unico objetivo fuera de tolerancia; la inconsistencia esta
   en la Tabla 4 del propio paper, no en la replicacion (seccion 10).

### Que NO se hizo, y por que

- **Optimizacion multiobjetivo (NSGA-II, MOPSO).** El paper lo sugiere en §3.3, pero
  exige frente de Pareto, metricas de hipervolumen y una interfaz distinta de
  `Optimizer`. Es una fase propia, no un agregado.
- **NPV, TIR, ROI, payback.** Exigen precio del H2, tarifa de inyeccion y prioridad de
  despacho, que el paper no publica.
- **Capacidad total variable y perfil de carga de H2** (§3.3 del paper): fuera del
  alcance de esta fase.

---

## Referencias

- Li, J., Zhang, Y., Wang, S., et al. (2024). *Capacity optimization of a
  wind-photovoltaic-electrolysis-battery (WPEB) hybrid energy system for power and
  hydrogen generation.* Applied Energy.
- Mirjalili, S., Mirjalili, S. M., & Lewis, A. (2014). *Grey Wolf Optimizer.* Advances in
  Engineering Software, 69, 46-61.
- Clerc, M., & Kennedy, J. (2002). *The particle swarm -- explosion, stability, and
  convergence in a multidimensional complex space.* IEEE Transactions on Evolutionary
  Computation, 6(1), 58-73.
- Dhiman, G., & Kumar, V. (2017). *Spotted hyena optimizer: A novel bio-inspired based
  metaheuristic technique for engineering applications.* Advances in Engineering Software,
  114, 48-70.
- Mirjalili, S., & Lewis, A. (2016). *The Whale Optimization Algorithm.* Advances in
  Engineering Software, 95, 51-67.
- Arcuri, A., & Briand, L. (2011). *A practical guide for using statistical tests to
  assess randomized algorithms in software engineering.* ICSE 2011.
- Vargha, A., & Delaney, H. D. (2000). *A critique and improvement of the CL common
  language effect size statistics of McGraw and Wong.* Journal of Educational and
  Behavioral Statistics, 25(2), 101-132.
- NASA POWER: <https://power.larc.nasa.gov/> -- fuente del dato horario.